# Matrix elements of the gluon operator $O_3$: ratio fit and sGEVP fit, double and single ratio

One notebook for the pion and the proton, the forward frame and the off-forward symmetric frame, and two fit models. The design is `NOTEBOOK_DESIGN.md` and `NOTEBOOK_STRUCTURE.md` (same folder); the fit choices are `FIT_PLAN.md`.

| part | sections | what it does |
|---|---|---|
| 0 | 0 | settings: hadron, frame, which models run, data sets, folders; the functions used by several sections |
| R: ratio fit | R0 - R5 | settings, ratio data, ratio plots, fits, check of the fits, matrix elements |
| S: sGEVP fit | S0 - S5 | settings, $C(t)$ and the summed 3pt, $M_{\rm eff}(t)$, fits, check of the fits, matrix elements |
| I: ITD | I1 - I2 | double ratio and single ratio; plots of the ratio fits, the sGEVP, or both |

**How to run it.** Section 0 first, then the parts you want, top to bottom. Every section reads what the section before saved, so after a restart: section 0, the settings cell of the part (R0 or S0), then the section you want. The cells of part R start with `%%only_for ratio`, those of part S with `%%only_for sgevp`: they run only when that model is in `fit_models` (section 0), otherwise they print "skipped".

**Where the files go.** The merged 1504-cfg data of R1, S1 and S2 (about 5.8 GB per hadron and frame) go to `data_dir`/`{hadron}_{frame}_ncfg1504/` (section 0: `~/Desktop/1504_cfg_data`, outside the repo); the fits, results, plots go to `{hadron}_{frame}_ncfg1504/` in this folder.

**Shapes.** Every array has its shape in a comment, each axis as name:size with the sizes of the default run (1504 cfg, forward). An axis that differs between the two data sets shows both: `cfg:724|780`.

**The energy $E_0$** of the single ratio comes from `matrix_element_analysis_final_2pt.ipynb` (forward frame only); run it once per hadron before section I1 if you want the single ratio.

| section | time (13 workers) | memory |
|---|---|---|
| R1 | ~3 min forward, ~10 min sym (9 GEVP times) | ~10 GB |
| R3 | ~10 min per hadron (9 GEVP times x 2 models) | ~4 GB |
| S1 | ~2 min forward, ~6 min sym | ~10 GB |
| S3 | ~2 min with the fixed windows; ~20 min per svdcut with the B1 grid | ~4 GB |
| others | seconds to a minute | small |

## 0. Settings: hadron, frame, models, data, folders; the functions used by several sections

In [ ]:
#======== 0. settings, and the functions used by several sections ========
import os
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"      #one BLAS thread per process: the parallel fits run faster;  works only before numpy is imported (a fresh kernel)
os.environ["OMP_NUM_THREADS"] = "1"
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import gvar as gv
import lsqfit
from joblib import Parallel, delayed            #runs the fits in n_workers processes at once
from IPython import get_ipython

#---- what to analyse
hadron = "pion"                     #"pion" or "proton" (the proton files are projected to positive parity)
frame = "forward"                   #"forward": q = (0,0,0);  "sym": q = (2,0,0), (-2,0,0), (0,2,0), (0,-2,0), averaged
fit_models = ["ratio", "sgevp"]     #the parts that run: "ratio" = part R, "sgevp" = part S;  ["ratio"], ["sgevp"] or both

#---- the data:  every data set is read and they are joined along the cfg axis, in this order
#  "dir": its folder;  "ncfg": the number of cfgs in the names of its 2pt files;  "ff_ncfg": the same for its FF files
#  "cfg_shift": added to its cfg numbers, so that the cfg numbers of two streams stay apart;  "skip_cfg": cfgs left out
data_sets = [{"name": "delta_stream_e", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta",
              "ncfg": 724, "ff_ncfg": 800, "cfg_shift": 0, "skip_cfg": []},
             {"name": "frontier_stream_d", "dir": "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_frontier_streamd",
              "ncfg": 781, "ff_ncfg": 798, "cfg_shift": 100000, "skip_cfg": [3006]}]
Lt = 96
Ls = 32
hyp = 8                             #HYP steps of the gluon operator
#the gluon operator O3 = TXTX + TYTY - 2 XYXY:  in the FF files the components are in the order XY, YZ, XZ, XT, YT, ZT, so O3 = FF[3,3] + FF[4,4] - 2 FF[0,0]

#---- the six interpolators of the 2pt files:  (smear, gamma);  smear 0 = B = [4.5, 2.25, 0.8], 1 = U = [1.5, 1.5, 0];  gamma 0 = G5, 1 = G5G4, 2 = G3G5
op_basis = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
smear_names = ["B", "U"]
gamma_names = ["G5", "G5G4", "G3G5"]
op_labels = [smear_names[s] + gamma_names[g] for s, g in op_basis]        #["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"]

#---- running
n_workers = 13                      #fits that run at the same time (this Mac has 18 cores);  1 = one after the other, in this kernel
show_plots = True                   #True = every figure is also drawn here;  False = only saved (PNG files and one PDF per plot section)

#---- folders and file names
analysis_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_analysis"     #fits, results, plots
data_dir = "/Users/mcp3270/Desktop/1504_cfg_data"                                                #the merged 1504-cfg data of R1, S1, S2 (outside the repo)
pt2_name = "2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_{frame}_q{qx}_{qy}_{qz}_{pt2_type}_fb_tsep20_3072src_ncfg{ncfg}_{hadron}.h5"
ff_name = "FF_hyp0-8_w0-9_q{qx}_{qy}_{qz}_ncfg{ncfg}.h5"

#---- made from the settings above
ncfg = 0                            #the number of cfgs after skip_cfg:  1504
for data in data_sets:
    ncfg = ncfg + data["ncfg"] - len(data["skip_cfg"])
if frame == "forward":
    q_list = [(0, 0, 0)]
else:
    q_list = [(2, 0, 0), (-2, 0, 0), (0, 2, 0), (0, -2, 0)]
if hadron == "pion":
    hadron_tag = "pion"
else:
    hadron_tag = "proton_projected"
c_sign = {"pion": -1, "proton": +1}[hadron]      #the pion 2pt matrix is negative definite: times -1 it can be used in the GEVP
save_dir = os.path.join(analysis_dir, f"{hadron}_{frame}_ncfg{ncfg}")      #everything this notebook saves for this hadron and frame
run_tag = f"{frame}_ncfg{ncfg}_{hadron_tag}"                              #the end of every file name
data_save_dir = os.path.join(data_dir, f"{hadron}_{frame}_ncfg{ncfg}")   #the data files of this hadron and frame
for folder in ["ratio_fits", "sgevp_fits", "results", "plots"]:
    os.makedirs(os.path.join(save_dir, folder), exist_ok=True)
os.makedirs(data_save_dir, exist_ok=True)
plot_title = f"{hadron_tag}, {frame}, {ncfg} cfg"                          #the start of every plot title

#the files of each section:  fill in the {...} with .format(...);  the data files (R1, S1, S2) are in data_save_dir, the others in save_dir
ratio_data_name = "ratio_gevp{t0}-{t_ref}_" + run_tag + ".h5"                                       #R1
ratio_fit_name = "ratio_fits/{model}/ratio_fit_{model}_gevp{t0}-{t_ref}_pz{pz}_svd{svdcut:g}_" + run_tag + ".npz"   #R3
ratio_result_name = "results/ratio_{model}_svd{svdcut:g}_" + run_tag + ".h5"                              #R5
pt2_6x6_name = "pt2_6x6_jk_" + run_tag + ".h5"                                                       #S1
pt3_summed_6x6_name = "pt3_summed_6x6_jk_hyp" + str(hyp) + "_" + run_tag + ".h5"                     #S1
meff_name = "sgevp_meff_" + run_tag + ".h5"                                                          #S2
sgevp_fit_name = "sgevp_fits/t0{t0}/sgevp_fit_t0{t0}_pz{pz}_svd{svdcut:g}_t{first}-{last}_" + run_tag + ".npz"   #S3
sgevp_result_name = "results/sgevp_{variant}_svd{svdcut:g}_" + run_tag + ".h5"                            #S5
itd_name = "results/itd_{model}_" + run_tag + ".h5"                                                       #I1

assert hadron in ["pion", "proton"], f"hadron = {hadron}: use pion or proton"
assert frame in ["forward", "sym"], f"frame = {frame}: use forward or sym"
for model in fit_models:
    assert model in ["ratio", "sgevp"], f"fit_models: {model} is not ratio or sgevp"


#======== functions used by several sections ========
def jackknife(x):
    #per-cfg values x (cfg:1504, ...) -> jackknife samples (jk:1504, ...);  sample j = the mean of every cfg but cfg j
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def jk(x):
    #jackknife mean and error over the first axis:  error = sqrt((N - 1)/N sum_j (x_j - mean)^2)
    n = x.shape[0]
    mean = x.mean(axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((x - mean) ** 2, axis=0))
    return mean, err


def op_matrix(pt2, ops):
    #the 2pt matrix of the operators ops (names of op_labels):
    #pt2 (cfg, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, ...) -> (cfg, a:n, b:n, ...),  a = sink operator, b = source operator
    n = len(ops)
    Cij = np.zeros(pt2.shape[:1] + (n, n) + pt2.shape[5:], dtype=complex)
    for a, name_a in enumerate(ops):
        s_a, g_a = op_basis[op_labels.index(name_a)]
        for b, name_b in enumerate(ops):
            s_b, g_b = op_basis[op_labels.index(name_b)]
            Cij[:, a, b] = pt2[:, s_a, s_b, g_a, g_b]
    return Cij


def gevp(C_t, C_t0):
    #C(t) u = lambda C(t0) u for a stack of matrices C_t, C_t0 (..., a:n, b:n), e.g. one per jackknife sample
    #C(t0) = L L^dag (Cholesky);  the eigenvalues lambda and vectors v of L^-1 C(t) L^-dag;  u = L^-dag v, so that u^dag C(t0) u = 1
    #returns lambda (..., state:n) in increasing order and u (..., a:n, state:n):  u[..., :, k] belongs to lambda[..., k]
    L_inv = np.linalg.inv(np.linalg.cholesky(C_t0))                     # (..., a:n, b:n)
    L_inv_dag = np.conj(np.swapaxes(L_inv, -1, -2))                     # (..., a:n, b:n)
    M = L_inv @ C_t @ L_inv_dag                                         # (..., a:n, b:n)
    M = 0.5 * (M + np.conj(np.swapaxes(M, -1, -2)))                     #Hermitian part (it differs only by rounding)
    lam, v = np.linalg.eigh(M)                                          # (..., state:n), (..., a:n, state:n)
    u = L_inv_dag @ v                                                   # (..., a:n, state:n)
    return lam, u


def neglog(x):
    #the parameter "neglog(M)" of a fit:  M = -exp(neglog(M)) < 0, so the fit varies log(-M)
    return -gv.exp(x)


def fit_jackknife(y_jk, x, model, prior, prior_args, starts, names, svdcut, maxit, tol, fitter):
    #one fit in every jackknife sample (FIT_PLAN.md sections 2-4):
    #  y_jk (jk:1504, point:n): the data of every sample;  x: the x of every point (a dictionary of arrays of length n)
    #  model(x, p): the fit function;  prior(*prior_args): the prior (made again after every gv.switch_gvar)
    #  starts: the start values of the fit of the mean (a list of dictionaries);  names: the parameters that are kept
    #the fit of the mean uses the covariance of all cfgs and keeps the lowest chi2 of the starts;
    #jackknife sample j starts from it and uses the covariance of the other cfgs (rebuilt without cfg j)
    if not gv.BufferDict.has_distribution("neglog"):    #also needed in every worker process
        gv.BufferDict.add_distribution("neglog", neglog)
    n_jk, n_point = y_jk.shape
    #---- the fit of the mean
    y_mean = y_jk.mean(axis=0)                                          # (point:n)
    d = y_jk - y_mean                                                   # (jk:1504, point:n)
    cov_all = (n_jk - 1) / n_jk * (d.T @ d)                             # (point:n, point:n)  jackknife covariance of the mean
    best = None
    chi2_starts = []
    for p0 in starts:
        gv.switch_gvar()
        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_mean, cov_all)), fcn=model, prior=prior(*prior_args), p0=p0,
                                   svdcut=svdcut, maxit=maxit, tol=tol, fitter=fitter)
        chi2_starts.append(float(fit.chi2))
        if best is None or fit.chi2 < best["chi2"]:
            best = {"chi2": float(fit.chi2), "dof": int(fit.dof), "svdn": int(fit.svdn), "at_maxit": int(fit.stopping_criterion == 0),
                    "p0": {key: np.array(fit.pmean[key]) for key in fit.pmean}}
        gv.restore_gvar()
    #---- every jackknife sample:  cov_j = covariance of the per-cfg values without cfg j, divided by the number of cfgs - 1
    y_cfg = n_jk * y_mean - (n_jk - 1) * y_jk                           # (cfg:1504, point:n)  the per-cfg values, rebuilt from the samples
    y_c = y_cfg - y_cfg.mean(axis=0)                                    # (cfg:1504, point:n)
    Q = y_c.T @ y_c                                                     # (point:n, point:n)
    m = n_jk - 1
    param_jk = {}
    for name in names:
        param_jk[name] = []
    chi2_jk = np.zeros(n_jk)                                            # (jk:1504)
    n_at_maxit = best["at_maxit"]
    for j in range(n_jk):
        cov_j = (Q - (1 + 1 / m) * np.outer(y_c[j], y_c[j])) / (m - 1) / (n_jk - 1)     # (point:n, point:n)
        gv.switch_gvar()
        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk[j], cov_j)), fcn=model, prior=prior(*prior_args), p0=best["p0"],
                                   svdcut=svdcut, maxit=maxit, tol=tol, fitter=fitter)
        for name in names:
            param_jk[name].append(np.array(fit.pmean[name]))
        chi2_jk[j] = fit.chi2
        n_at_maxit = n_at_maxit + int(fit.stopping_criterion == 0)
        gv.restore_gvar()
    for name in names:
        param_jk[name] = np.array(param_jk[name])                       # (jk:1504, parameter:n_name)
    #---- the fitted curve (mean over the samples) and the pulls:  (data mean - curve) / jackknife error of the point
    curve_jk = np.zeros((n_jk, n_point))                                # (jk:1504, point:n)
    for j in range(n_jk):
        p_j = {}
        for name in names:
            p_j[name] = param_jk[name][j]
        curve_jk[j] = model(x, p_j)
    y_m, y_e = jk(y_jk)                                                 # (point:n), (point:n)
    curve_mean = curve_jk.mean(axis=0)                                  # (point:n)
    return {"param_jk": param_jk, "chi2_jk": chi2_jk, "chi2": best["chi2"], "dof": best["dof"], "svdn": best["svdn"],
            "n_at_maxit": n_at_maxit, "chi2_starts": np.array(chi2_starts), "y_mean": y_m, "y_err": y_e,
            "curve_mean": curve_mean, "pull": (y_m - curve_mean) / y_e}


def aic_average(M_choice, dE_choice, chi2_jk, k, n_cut):
    #the average over several fits ("choices") in every jackknife sample j (FIT_PLAN.md sections 7 and 9):
    #  aic = chi2_j + 2 k + 2 N_cut of each choice;  weight = exp[-(aic - smallest aic)/2], normalized to sum 1
    #M_choice (jk:1504, choice:n, w:10), dE_choice (jk:1504, choice:n), chi2_jk (jk:1504, choice:n), k (choice:n), n_cut (choice:n)
    aic = chi2_jk + 2 * k + 2 * n_cut                                   # (jk:1504, choice:n)
    weight = np.exp(-0.5 * (aic - aic.min(axis=1, keepdims=True)))      # (jk:1504, choice:n)
    weight = weight / weight.sum(axis=1, keepdims=True)
    M = (weight[:, :, None] * M_choice).sum(axis=1)                     # (jk:1504, w:10)
    dE = (weight * dE_choice).sum(axis=1)                               # (jk:1504)
    return M, dE, weight


def save_figure(fig, pdf, folder, name):
    #a figure of a plot section:  saved as PNG in plots/folder and as a page of the open PDF pdf, then drawn here (show_plots) or closed
    os.makedirs(os.path.join(save_dir, "plots", folder), exist_ok=True)
    fig.savefig(os.path.join(save_dir, "plots", folder, f"{name}_{run_tag}.png"), dpi=110)
    pdf.savefig(fig)
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


def only_for(line, cell):
    #the first line "%%only_for ratio" (or "%%only_for sgevp") of a cell of part R (or S):
    #the cell runs only when that model is in fit_models;  otherwise it prints "skipped"
    model = line.strip()
    if model not in fit_models:
        print(f"skipped: this section is for the {model} model, and fit_models = {fit_models}")
        return
    result = get_ipython().run_cell(cell)
    if result.error_in_exec is not None:
        raise RuntimeError("this section stopped with the error above")


get_ipython().register_magic_function(only_for, magic_kind="cell")

#---- print what was chosen
for data in data_sets:
    print(f"data set {data['name']}: {data['ncfg']} cfg, {len(data['skip_cfg'])} left out, cfg numbers + {data['cfg_shift']}")
print(f"{hadron_tag} | {frame}, q {q_list} | {ncfg} cfg | models {fit_models} | {n_workers} workers")
print("folder", save_dir)
print("data folder", data_save_dir)

## R0. Ratio fit: settings (GEVP bases and times, fit windows and priors) and the model functions

In [ ]:
%%only_for ratio
#======== R0. settings of the ratio fit, and its model functions ========
#every choice of part R is here, so that any section of part R can be run again after section 0 and this cell

#---- the GEVP of the ratio (R1):  the operators of each pz, and the GEVP times (t0, t_ref)
#forward:  the default bases (FIT_PLAN.md section 3);  sym:  the bases of 2026-11-12 (FIT_PLAN.md section 15)
pion_forward_basis = {0: ["BG5G4", "UG5", "UG5G4"], 1: ["UG5", "UG5G4"], 2: ["BG5", "UG5G4", "UG3G5"], 3: ["UG3G5"],
                      4: ["UG3G5"], 5: ["BG5G4", "UG5G4"], 6: ["BG3G5", "UG3G5"]}
proton_forward_basis = {0: ["UG5", "UG5G4"], 1: ["UG5", "UG5G4"], 2: ["UG5G4", "UG3G5"], 3: ["UG5G4", "UG3G5"],
                        4: ["UG5G4", "UG3G5"], 5: ["UG5G4"], 6: ["BG3G5"]}
pion_sym_basis = {0: ["BG3G5", "UG5G4", "UG3G5"], 1: ["UG5G4"], 2: ["UG5G4"], 3: ["BG3G5", "UG5G4"],
                  4: ["BG5G4", "BG3G5", "UG5"], 5: ["BG3G5"], 6: ["BG3G5"]}
proton_sym_basis = {0: ["BG5", "UG5"], 1: ["BG5", "UG5"], 2: ["BG5"], 3: ["UG5G4", "UG3G5"],
                    4: ["UG5"], 5: ["UG3G5"], 6: ["BG3G5"]}
#the final result averages over these nine (t0, t_ref) (FIT_PLAN.md section 9);  [(3, 5)] for one
ratio_gevp_times = [(2, 3), (2, 4), (2, 5), (3, 4), (3, 5), (3, 6), (4, 5), (4, 6), (4, 7)]

#---- the fits (R3)
ratio_fit_models = ["separate", "shared"]       #"separate": Ai, Afi of every w;  "shared": one Ai, Afi for all w (fraction of M00)
ratio_fit_svdcuts = [1e-2]
#per pz:  "window" = (first tsep, last tsep, tau_skip);  "dE_prior", "M00_prior" = (center, width)
pion_forward_fit = {0: {"window": (6, 15, 3), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                    1: {"window": (5, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                    2: {"window": (4, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                    3: {"window": (4, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                    4: {"window": (4, 12, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                    5: {"window": (4, 9, 2), "dE_prior": (0.3, 0.5), "M00_prior": (-1.0, 100.0)},
                    6: {"window": (4, 9, 2), "dE_prior": (0.3, 0.5), "M00_prior": (-1.0, 100.0)}}
proton_forward_fit = {0: {"window": (7, 14, 3), "dE_prior": (0.5, 1.0), "M00_prior": (-1.0, 10.0)},
                      1: {"window": (6, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                      2: {"window": (6, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                      3: {"window": (6, 13, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                      4: {"window": (4, 10, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                      5: {"window": (4, 9, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)},
                      6: {"window": (4, 10, 2), "dE_prior": (0.5, 0.5), "M00_prior": (-1.0, 10.0)}}
pion_sym_fit = pion_forward_fit                 #the same windows and priors
proton_sym_fit = {0: {"window": (6, 10, 3), "dE_prior": (0.5, 1.0), "M00_prior": (-1.0, 10.0)},       #pz 0: the refit of 2026-11-12
                  1: proton_forward_fit[1], 2: proton_forward_fit[2], 3: proton_forward_fit[3],
                  4: proton_forward_fit[4], 5: proton_forward_fit[5], 6: proton_forward_fit[6]}
ratio_A_prior = (0.0, 1e4)                      #Ai and Afi
ratio_dE_starts = [0.15, 0.3, 0.5, 0.8, 1.2]    #the starts of the fit of the mean: one per dE, with M00 = -1, Ai = Afi = 0
ratio_maxit = 20000
ratio_tol = 1e-8
ratio_fitter = "scipy_least_squares"
#fixed:  the ratio is folded (R(tsep, tau) and R(tsep, tsep - tau) averaged), tau from tau_skip to tsep/2;  all w in one fit with one gap dE;
#        M00 < 0 (the fit varies log(-M00));  log(dE);  full covariance of all points;  every jackknife sample with its own covariance

#---- the dictionaries of this hadron and frame
if hadron == "pion" and frame == "forward":
    ratio_basis, ratio_fit_choice = pion_forward_basis, pion_forward_fit
elif hadron == "pion":
    ratio_basis, ratio_fit_choice = pion_sym_basis, pion_sym_fit
elif frame == "forward":
    ratio_basis, ratio_fit_choice = proton_forward_basis, proton_forward_fit
else:
    ratio_basis, ratio_fit_choice = proton_sym_basis, proton_sym_fit
for pz in ratio_basis:
    for name in ratio_basis[pz]:
        assert name in op_labels, f"ratio basis of pz {pz}: {name} is not one of {op_labels}"
for model in ratio_fit_models:
    assert model in ["separate", "shared"], f"ratio_fit_models: {model} is not separate or shared"


def ratio_points(window):
    #the (tsep, tau) of the folded ratio in a fit window (first tsep, last tsep, tau_skip):  tau = tau_skip .. tsep/2 at every tsep
    first, last, tau_skip = window
    tsep = []
    tau = []
    for t in range(first, last + 1):
        for a in range(tau_skip, t // 2 + 1):
            tsep.append(t)
            tau.append(a)
    return np.array(tsep), np.array(tau)


def ratio_model_separate(x, p):
    #R_w(tsep, tau) = M00_w + Ai_w [e^(-dE tau) + e^(-dE (tsep - tau))] + Afi_w e^(-dE tsep);  x["iw"] = the w index of every point
    dE = p["dE"][0]
    e1 = gv.exp(-dE * x["tau"]) + gv.exp(-dE * (x["tsep"] - x["tau"]))
    e2 = gv.exp(-dE * x["tsep"])
    return p["M00"][x["iw"]] + p["Ai"][x["iw"]] * e1 + p["Afi"][x["iw"]] * e2


def ratio_model_shared(x, p):
    #R_w(tsep, tau) = M00_w [1 + Ai (e^(-dE tau) + e^(-dE (tsep - tau))) + Afi e^(-dE tsep)]:  one Ai, Afi for all w
    dE = p["dE"][0]
    e1 = gv.exp(-dE * x["tau"]) + gv.exp(-dE * (x["tsep"] - x["tau"]))
    e2 = gv.exp(-dE * x["tsep"])
    return p["M00"][x["iw"]] * (1 + p["Ai"][0] * e1 + p["Afi"][0] * e2)


def ratio_prior(model, pz, n_w):
    #log(-M00) = log(1(10)) = 0(10) for M00_prior (-1, 10);  Ai, Afi;  log(dE)
    choice = ratio_fit_choice[pz]
    n_amp = n_w if model == "separate" else 1
    p = gv.BufferDict()
    p["neglog(M00)"] = gv.log(-gv.gvar(np.full(n_w, float(choice["M00_prior"][0])), np.full(n_w, float(choice["M00_prior"][1]))))
    p["Ai"] = gv.gvar(np.full(n_amp, ratio_A_prior[0]), np.full(n_amp, ratio_A_prior[1]))
    p["Afi"] = gv.gvar(np.full(n_amp, ratio_A_prior[0]), np.full(n_amp, ratio_A_prior[1]))
    p["log(dE)"] = gv.log(gv.gvar([float(choice["dE_prior"][0])], [float(choice["dE_prior"][1])]))
    return p


def ratio_settings_text(model, pz, svdcut):
    #every setting of one fit, as text:  saved in its file;  R3 does not redo a fit whose file has the same text
    choice = ratio_fit_choice[pz]
    return (f"{model}, window {choice['window']}, folded, dE {choice['dE_prior']}, M00 {choice['M00_prior']} (M00 < 0), Ai Afi {ratio_A_prior}, "
            f"starts {ratio_dE_starts}, svdcut {svdcut:g}, maxit {ratio_maxit}, tol {ratio_tol}, {ratio_fitter}, covariance per sample")


print(f"{hadron_tag}, {frame}:  GEVP times {ratio_gevp_times};  models {ratio_fit_models};  svdcuts {ratio_fit_svdcuts}")
for pz in ratio_basis:
    choice = ratio_fit_choice[pz]
    print(f"pz {pz}: {' '.join(ratio_basis[pz]):24s}  window {choice['window']}, dE {choice['dE_prior']}, M00 {choice['M00_prior']}")

## R1. Ratio data: GEVP of each pz, projected 2pt, method-1 3pt, ratio $R(t_{\rm sep}, \tau)$

In [ ]:
%%only_for ratio
#======== R1. ratio data:  GEVP of each pz, projected 2pt, method-1 3pt, ratio R(tsep, tau);  one file per GEVP time ========
#(the steps of matrix_element_analysis_fit_per_pz.ipynb sections 1, 3 and 4, for every (t0, t_ref) of ratio_gevp_times)
#GEVP:  C(t_ref) v = lambda C(t0) v on the cfg mean of the 2pt matrix of this pz's operators (c_sign, Hermitian part);  v0 = the ground state (largest lambda)
#  forward:  the 2pt of the C3input pack;  sym:  [C2pf(q) + C2pf(-q)]/2, one vector per q, used at the sink and at the source
#3pt, method 1:  [v0^dag C3input v0](t_src, tsep) x O3(t_src + tau) (backward half: t_src - tau), mean over the sources,
#  minus jackknife(the same 2pt, mean over the sources) x jackknife(O3 on the same slices, mean over the sources)
#ratio:  R = Re 3pt / C2f(tsep) x sqrt[C2i(tsep - tau) C2f(tau) C2f(tsep) / (C2f(tsep - tau) C2i(tau) C2i(tsep))],
#  C2f = projected 2pt at q, C2i = at -q;  the 3pt and the 2pt are averaged over the two halves and the q first;  forward: the root is 1
time_start = time.time()
half_sign = {"forward": +1, "backward": -1}     #the operator sits at t_src + tau (forward half) or t_src - tau (backward half)
if frame == "forward":
    gevp_pack = "C3input"       #at q = 0 the C3input pack is also the 2pt at p_f and p_i
else:
    gevp_pack = "C2pf"          #C2pi(q) = C2pf(-q): the C2pi packs are not needed

#---- O3 of every cfg and q
O3 = {}                         #O3[q]: (cfg:1504, w:10, t:96)
for q in q_list:
    O3_parts = []
    for data in data_sets:
        file_2pt = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input",
                                                             ncfg=data["ncfg"], hadron=hadron_tag))
        with h5py.File(file_2pt, "r") as f:
            cfg_d = f["cfg_list"][()]                                       # (cfg:724|781)
        cfg_d = cfg_d[~np.isin(cfg_d, data["skip_cfg"])]                    # (cfg:724|780)  the 2pt cfgs of this data set
        file_ff = os.path.join(data["dir"], ff_name.format(qx=q[0], qy=q[1], qz=q[2], ncfg=data["ff_ncfg"]))
        print("reading", file_ff)
        with h5py.File(file_ff, "r") as f:
            assert tuple(f["q"][()]) == q, f"{file_ff}: another q"
            cfg_ff = f["cfg_list"][()]                                      # (cfg:800|798)
            idx = np.searchsorted(cfg_ff, cfg_d)                            # (cfg:724|780)  where the 2pt cfgs are in the FF file
            assert np.array_equal(cfg_ff[idx], cfg_d), f"{data['name']}: some 2pt cfgs are not in the FF file"
            ihyp = list(f["hyp_list"][()]).index(hyp)
            TXTX = f["FF"][idx, 3, 3, ihyp]                                 # (cfg:724|780, w:10, t:96)
            TYTY = f["FF"][idx, 4, 4, ihyp]                                 # (cfg:724|780, w:10, t:96)
            XYXY = f["FF"][idx, 0, 0, ihyp]                                 # (cfg:724|780, w:10, t:96)
        O3_parts.append(TXTX + TYTY - 2 * XYXY)
    O3[q] = np.concatenate(O3_parts)                                        # (cfg:1504, w:10, t:96)

#---- the 2pt of the GEVP:  every q and half, mean over the sources
C2_gevp = {}                    #C2_gevp[(q, half)]: (cfg:1504, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, pz:7, t:20)
cfg_list = None
for q in q_list:
    for half in half_sign:
        pt2_parts = []
        cfg_parts = []
        for data in data_sets:
            file_2pt = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type=gevp_pack,
                                                                 ncfg=data["ncfg"], hadron=hadron_tag))
            print("reading", file_2pt, half)
            with h5py.File(file_2pt, "r") as f:
                cfg_d = f["cfg_list"][()]                                   # (cfg:724|781)
                pz_list = f["pz_list"][()]                                  # (pz:7)
                # pt2_d: (cfg:724|781, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, t_src:24, pz:7, t:20)
                pt2_d = f[f"pt2_{half}"][()]
            keep = ~np.isin(cfg_d, data["skip_cfg"])                        # (cfg:724|781)
            pt2_parts.append(pt2_d[keep].mean(axis=5))                      # (cfg:724|780, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, pz:7, t:20)
            cfg_parts.append(cfg_d[keep] + data["cfg_shift"])
        C2_gevp[(q, half)] = np.concatenate(pt2_parts)
        if cfg_list is None:
            cfg_list = np.concatenate(cfg_parts)                            # (cfg:1504)
        assert np.array_equal(np.concatenate(cfg_parts), cfg_list), f"q {q}, {half}: other cfgs than the first file"
n_cfg = len(cfg_list)
n_pz = len(pz_list)
n_tsep = C2_gevp[(q_list[0], "forward")].shape[-1]
assert n_cfg == ncfg and len(np.unique(cfg_list)) == n_cfg, "the cfg numbers of the data sets overlap, or skip_cfg is wrong"

#---- the GEVP of every (t0, t_ref), half, q and pz:  v0, and the projected 2pt of every cfg
v0 = {}                         #v0[(t0, t_ref, half, q)]: (pz:7, a:6), zero for the operators this pz does not use
C2_rec = {}                     #C2_rec[(t0, t_ref, half, q)]: (cfg:1504, pz:7, t:20), c_sign v0^dag C v0
for t0, t_ref in ratio_gevp_times:
    for half in half_sign:
        for q in q_list:
            q_minus = (-q[0], -q[1], -q[2])
            v0[(t0, t_ref, half, q)] = np.zeros((n_pz, len(op_labels)), dtype=complex)
            C2_rec[(t0, t_ref, half, q)] = np.zeros((n_cfg, n_pz, n_tsep), dtype=complex)
            for ipz in range(n_pz):
                ops = ratio_basis[int(pz_list[ipz])]
                use = [op_labels.index(name) for name in ops]
                C_plus = op_matrix(C2_gevp[(q, half)][..., ipz, :], ops)            # (cfg:1504, a:n, b:n, t:20)
                C_minus = op_matrix(C2_gevp[(q_minus, half)][..., ipz, :], ops)     # (cfg:1504, a:n, b:n, t:20)
                C_mean = c_sign * 0.5 * (C_plus + C_minus).mean(axis=0)             # (a:n, b:n, t:20)
                C_mean = 0.5 * (C_mean + C_mean.conj().swapaxes(0, 1))              #Hermitian part
                lam, u = gevp(C_mean[:, :, t_ref], C_mean[:, :, t0])                # (state:n), (a:n, state:n)
                v0[(t0, t_ref, half, q)][ipz, use] = u[:, -1]                       #the largest lambda: the ground state
                v = u[:, -1]                                                        # (a:n)
                C2_rec[(t0, t_ref, half, q)][:, ipz] = c_sign * np.einsum("a,cabt,b->ct", v.conj(), C_plus, v)
del C2_gevp

#---- the 3pt of every (t0, t_ref):  summed over the halves and the q, divided by their number at the end
pt3_sum = {}                    #pt3_sum[(t0, t_ref)]: (jk:1504, w:10, pz:7, tsep:20, tau:20), Re 3pt
c2f_sum = {}                    #c2f_sum[(t0, t_ref)]: (jk:1504, pz:7, tsep:20), C2 at q
c2i_sum = {}                    #c2i_sum[(t0, t_ref)]: (jk:1504, pz:7, tsep:20), C2 at -q
n_w = O3[q_list[0]].shape[1]
tau_list = np.arange(n_tsep)
for t0, t_ref in ratio_gevp_times:
    pt3_sum[(t0, t_ref)] = np.zeros((n_cfg, n_w, n_pz, n_tsep, n_tsep))
    c2f_sum[(t0, t_ref)] = np.zeros((n_cfg, n_pz, n_tsep))
    c2i_sum[(t0, t_ref)] = np.zeros((n_cfg, n_pz, n_tsep))
i_cfg = np.arange(n_cfg)[:, None, None]
for half in half_sign:
    for q in q_list:
        q_minus = (-q[0], -q[1], -q[2])
        #read the C3input pack of q, every source
        pt2_parts = []
        t_src_parts = []
        for data in data_sets:
            file_2pt = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input",
                                                                 ncfg=data["ncfg"], hadron=hadron_tag))
            print("reading", file_2pt, half)
            with h5py.File(file_2pt, "r") as f:
                cfg_d = f["cfg_list"][()]                                   # (cfg:724|781)
                # pt2_d: (cfg:724|781, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, t_src:24, pz:7, t:20)
                pt2_d = f[f"pt2_{half}"][()]
                t_src_d = f["t_src_list"][()]                               # (cfg:724|781, t_src:24)  the slice of every source
            keep = ~np.isin(cfg_d, data["skip_cfg"])                        # (cfg:724|781)
            pt2_parts.append(pt2_d[keep])
            t_src_parts.append(t_src_d[keep])
            del pt2_d
        # pt2_src: (cfg:1504, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, t_src:24, pz:7, t:20)
        pt2_src = np.concatenate(pt2_parts)
        del pt2_parts
        t_src_list = np.concatenate(t_src_parts)                            # (cfg:1504, t_src:24)
        n_tsrc = t_src_list.shape[1]
        #O3 on the slices of every source
        t_op = (t_src_list[:, :, None] + half_sign[half] * tau_list) % Lt  # (cfg:1504, t_src:24, tau:20)
        O_tau = O3[q][i_cfg, :, t_op]                                       # (cfg:1504, t_src:24, tau:20, w:10)  numpy puts w last here
        O_mean_jk = jackknife(O_tau.mean(axis=1))                           # (jk:1504, tau:20, w:10)
        for t0, t_ref in ratio_gevp_times:
            for ipz in range(n_pz):
                ops = ratio_basis[int(pz_list[ipz])]
                use = [op_labels.index(name) for name in ops]
                v = v0[(t0, t_ref, half, q)][ipz, use]                      # (a:n)
                Cij = op_matrix(pt2_src[..., ipz, :], ops)                  # (cfg:1504, a:n, b:n, t_src:24, tsep:20)
                C3 = c_sign * np.einsum("a,cabst,b->cst", v.conj(), Cij, v)  # (cfg:1504, t_src:24, tsep:20)  v0^dag C3input v0 of every source
                connected = np.einsum("cst,csuw->cwtu", C3, O_tau, optimize=True) / n_tsrc     # (cfg:1504, w:10, tsep:20, tau:20)
                vacuum = np.einsum("ct,cuw->cwtu", jackknife(C3.mean(axis=1)), O_mean_jk, optimize=True)     # (jk:1504, w:10, tsep:20, tau:20)
                pt3_sum[(t0, t_ref)][:, :, ipz] += (jackknife(connected) - vacuum).real
            c2f_sum[(t0, t_ref)] += jackknife(C2_rec[(t0, t_ref, half, q)]).real
            c2i_sum[(t0, t_ref)] += jackknife(C2_rec[(t0, t_ref, half, q_minus)]).real
        del pt2_src, O_tau
        print(f"  {half}, q {q}: 3pt of every GEVP time and pz done, {time.time() - time_start:.0f} s")

#---- the ratio of every (t0, t_ref), saved
tsep_grid, tau_grid = np.meshgrid(np.arange(n_tsep), tau_list, indexing="ij")      # (tsep:20, tau:20), (tsep:20, tau:20)
in_window = tau_grid <= tsep_grid                                                   # (tsep:20, tau:20)
rest_grid = np.where(in_window, tsep_grid - tau_grid, 0)                            # (tsep:20, tau:20)  tsep - tau (0 where R is NaN)
n_sum = 2 * len(q_list)                                                             #the number of (half, q) in the sums
for t0, t_ref in ratio_gevp_times:
    pt3 = pt3_sum[(t0, t_ref)] / n_sum                                              # (jk:1504, w:10, pz:7, tsep:20, tau:20)
    c2f = c2f_sum[(t0, t_ref)] / n_sum                                              # (jk:1504, pz:7, tsep:20)
    c2i = c2i_sum[(t0, t_ref)] / n_sum                                              # (jk:1504, pz:7, tsep:20)
    root = np.sqrt(c2i[:, :, rest_grid] * c2f[:, :, tau_grid] * c2f[:, :, tsep_grid]
                   / (c2f[:, :, rest_grid] * c2i[:, :, tau_grid] * c2i[:, :, tsep_grid]))      # (jk:1504, pz:7, tsep:20, tau:20)
    ratio = np.where(in_window, pt3 * (root / c2f[:, :, tsep_grid])[:, None], np.nan)          # (jk:1504, w:10, pz:7, tsep:20, tau:20)
    file = os.path.join(data_save_dir, ratio_data_name.format(t0=t0, t_ref=t_ref))
    if os.path.exists(file):
        print("replacing", file)
    with h5py.File(file, "w") as f:
        f["ratio_jk"] = ratio                                   # (jk:1504, w:10, pz:7, tsep:20, tau:20)  Re R, NaN for tau > tsep
        f["C2_jk"] = c2f                                        # (jk:1504, pz:7, tsep:20)  the projected 2pt, halves and q averaged
        for half in half_sign:
            f[f"v0_{half}"] = np.array([v0[(t0, t_ref, half, q)] for q in q_list])     # (q:1, pz:7, a:6)
        f["q_list"] = q_list
        f["t0"] = t0
        f["t_ref"] = t_ref
        f["basis"] = [" ".join(ratio_basis[int(pz)]) for pz in pz_list]                 # (pz:7)  the operators of every pz
        f["cfg_list"] = cfg_list
        f["pz_list"] = pz_list
        f["op_labels"] = op_labels
        f["hyp"] = hyp
        f["data_sets"] = [data["name"] for data in data_sets]
        f["data_set_skip_cfg"] = [" ".join(str(c) for c in data["skip_cfg"]) for data in data_sets]
    mean, err = jk(ratio[:, 0, :, 8, 4])                                                # (pz:7), (pz:7)
    print(f"t0 {t0}, t_ref {t_ref}: Re R at w 0, tsep 8, tau 4, pz 0-{n_pz - 1}: " + " ".join(f"{m:+.3f}({e * 1e3:.0f})" for m, e in zip(mean, err)))
del pt3_sum
print(f"saved {len(ratio_gevp_times)} ratio files in {data_save_dir};  {time.time() - time_start:.0f} s")

## R2. Plots of the ratio

In [ ]:
%%only_for ratio
#======== R2. plots of the ratio:  R against tau - tsep/2, one color per tsep;  one page per pz, one panel per w;  no fit ========
plot_gevp_time = (3, 5)                         #(t0, t_ref) of the ratio that is plotted
plot_w = [0, 2, 4, 6, 8]
plot_tsep = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
plot_err_max = 0.3                              #points with a larger error are left out, so that noisy tsep do not set the y range
tsep_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948", "#8c564b", "#17becf", "#7f7f7f", "#bcbd22"]

file = os.path.join(data_save_dir, ratio_data_name.format(t0=plot_gevp_time[0], t_ref=plot_gevp_time[1]))
assert os.path.exists(file), f"no ratio file {file}:  run R1 with {plot_gevp_time} in ratio_gevp_times"
with h5py.File(file, "r") as f:
    ratio = f["ratio_jk"][()]                   # (jk:1504, w:10, pz:7, tsep:20, tau:20)
    pz_list = f["pz_list"][()]                  # (pz:7)
    basis = list(f["basis"].asstr()[()])        # (pz:7)  the operators of every pz
ratio_mean, ratio_err = jk(ratio)               # (w:10, pz:7, tsep:20, tau:20), (w:10, pz:7, tsep:20, tau:20)
del ratio

pdf = PdfPages(os.path.join(save_dir, "plots", f"R2_ratio_gevp{plot_gevp_time[0]}-{plot_gevp_time[1]}_{run_tag}.pdf"))
for ipz, pz in enumerate(pz_list):
    fig, axes = plt.subplots(1, len(plot_w), figsize=(3.2 * len(plot_w), 3.6), sharey=True)
    for i_panel, w in enumerate(plot_w):
        ax = axes[i_panel]
        for i_tsep, tsep in enumerate(plot_tsep):
            tau = np.arange(1, tsep)                                        #tau = 1 .. tsep - 1
            m = ratio_mean[w, ipz, tsep, tau]
            e = ratio_err[w, ipz, tsep, tau]
            ok = e < plot_err_max
            shift = 0.06 * (i_tsep - (len(plot_tsep) - 1) / 2)              #a small shift, so that the error bars do not sit on top of each other
            ax.errorbar(tau[ok] - tsep / 2 + shift, m[ok], e[ok], color=tsep_colors[i_tsep % len(tsep_colors)], marker="o", ms=3,
                        ls="none", elinewidth=0.8, capsize=1.5, label=f"tsep {tsep}")
        ax.axhline(0, color="#898781", lw=0.6)
        ax.grid(color="#e1e0d9", lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"w = {w}", fontsize=9)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$", fontsize=8)
    axes[0].set_ylabel("Re R", fontsize=9)
    axes[-1].legend(fontsize=6, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.suptitle(f"{plot_title}: ratio, pz {pz}, operators {basis[ipz]}, GEVP t0 {plot_gevp_time[0]}, t_ref {plot_gevp_time[1]};  "
                 f"points with error > {plot_err_max} left out", fontsize=9.5)
    fig.tight_layout()
    save_figure(fig, pdf, "R2_ratio", f"ratio_pz{pz}")
pdf.close()

## R3. Ratio fits: every model, GEVP time, pz and svdcut

In [ ]:
%%only_for ratio
#======== R3. ratio fits:  every model, GEVP time, pz and svdcut of R0;  one file per fit;  a fit whose file is saved with the same settings is not done again ========
#the model, the windows and the priors are in R0;  the fit itself is fit_jackknife (section 0)


def ratio_fit_one(model, t0, t_ref, pz, svdcut):
    #one ratio fit in every jackknife sample;  saves its file and returns one line of text
    time_fit = time.time()
    with h5py.File(os.path.join(data_save_dir, ratio_data_name.format(t0=t0, t_ref=t_ref)), "r") as f:
        ipz = list(f["pz_list"][()]).index(pz)
        ratio = f["ratio_jk"][:, :, ipz]                                    # (jk:1504, w:10, tsep:20, tau:20)
    n_jk, n_w = ratio.shape[:2]
    #---- the folded ratio on the points of the window, all w after each other
    tsep, tau = ratio_points(ratio_fit_choice[pz]["window"])                # (point_w:n), (point_w:n)  the points of one w
    y = 0.5 * (ratio[:, :, tsep, tau] + ratio[:, :, tsep, tsep - tau])     # (jk:1504, w:10, point_w:n)
    assert not np.isnan(y).any(), f"pz {pz}: NaN in the ratio inside the window {ratio_fit_choice[pz]['window']}"
    y_jk = y.reshape(n_jk, n_w * len(tsep))                                 # (jk:1504, point:10 n)
    x = {"tsep": np.tile(tsep.astype(float), n_w), "tau": np.tile(tau.astype(float), n_w),
         "iw": np.repeat(np.arange(n_w), len(tsep))}                        #the tsep, tau and w index of every point
    #---- the starts of the fit of the mean:  one per dE of ratio_dE_starts, with M00 = -1, Ai = Afi = 0
    n_amp = n_w if model == "separate" else 1
    starts = []
    for dE_start in ratio_dE_starts:
        starts.append({"neglog(M00)": np.zeros(n_w), "Ai": np.zeros(n_amp), "Afi": np.zeros(n_amp), "log(dE)": np.log([dE_start])})
    if model == "separate":
        fcn = ratio_model_separate
    else:
        fcn = ratio_model_shared
    result = fit_jackknife(y_jk, x, fcn, ratio_prior, (model, pz, n_w), starts, ["M00", "Ai", "Afi", "dE"],
                           svdcut, ratio_maxit, ratio_tol, ratio_fitter)
    #---- save
    file = os.path.join(save_dir, ratio_fit_name.format(model=model, t0=t0, t_ref=t_ref, pz=pz, svdcut=svdcut))
    np.savez(file, M_jk=result["param_jk"]["M00"], Ai_jk=result["param_jk"]["Ai"], Afi_jk=result["param_jk"]["Afi"],
             dE_jk=result["param_jk"]["dE"][:, 0], chi2_jk=result["chi2_jk"], chi2=result["chi2"], dof=result["dof"], svdn=result["svdn"],
             n_at_maxit=result["n_at_maxit"], chi2_starts=result["chi2_starts"], y_mean=result["y_mean"], y_err=result["y_err"],
             curve_mean=result["curve_mean"], pull=result["pull"], tsep=tsep, tau=tau, k=3 * n_w + 1 if model == "separate" else n_w + 3,
             n_point=len(y_jk[0]), window=np.array(ratio_fit_choice[pz]["window"]), svdcut=svdcut, n_jk=n_jk,
             settings=ratio_settings_text(model, pz, svdcut))
    return f"{os.path.basename(file)}: chi2/dof {result['chi2'] / result['dof']:.3f}, {result['n_at_maxit']} fits at maxit, {time.time() - time_fit:.0f} s"


#---- the fits that are not saved yet
tasks = []                      #(model, t0, t_ref, pz, svdcut)
n_all = 0
for model in ratio_fit_models:
    os.makedirs(os.path.join(save_dir, "ratio_fits", model), exist_ok=True)
    for t0, t_ref in ratio_gevp_times:
        assert os.path.exists(os.path.join(data_save_dir, ratio_data_name.format(t0=t0, t_ref=t_ref))), f"no ratio file for t0 {t0}, t_ref {t_ref}: run R1"
        for pz in ratio_fit_choice:
            for svdcut in ratio_fit_svdcuts:
                n_all = n_all + 1
                file = os.path.join(save_dir, ratio_fit_name.format(model=model, t0=t0, t_ref=t_ref, pz=pz, svdcut=svdcut))
                if os.path.exists(file):
                    saved = str(np.load(file)["settings"])
                    assert saved == ratio_settings_text(model, pz, svdcut), (f"{file} was fitted with other settings:\n  {saved}\n"
                                                                            f"delete the file to fit it again with the settings of R0")
                else:
                    tasks.append((model, t0, t_ref, pz, svdcut))
print(f"{n_all} fits in all, {n_all - len(tasks)} already saved, {len(tasks)} to do with {n_workers} workers", flush=True)
time_start = time.time()
lines = Parallel(n_jobs=n_workers, verbose=5)(delayed(ratio_fit_one)(*task) for task in tasks)
print(f"done, {time.time() - time_start:.0f} s")

#---- a table of every fit:  chi2/dof of the fit of the mean, M00(w 0), dE, fits that stopped at maxit
for model in ratio_fit_models:
    for svdcut in ratio_fit_svdcuts:
        print(f"\n{model}, svdcut {svdcut:g}:   pz  (t0, t_ref)  chi2/dof   M00(w 0)        dE              at maxit")
        for pz in ratio_fit_choice:
            for t0, t_ref in ratio_gevp_times:
                d = np.load(os.path.join(save_dir, ratio_fit_name.format(model=model, t0=t0, t_ref=t_ref, pz=pz, svdcut=svdcut)))
                M0_mean, M0_err = jk(d["M_jk"][:, 0])
                dE_mean, dE_err = jk(d["dE_jk"])
                print(f"{'':28s}{pz}   ({t0}, {t_ref})     {float(d['chi2']) / int(d['dof']):6.3f}    {str(gv.gvar(M0_mean, M0_err)):14s}  "
                      f"{str(gv.gvar(dE_mean, dE_err)):14s}  {int(d['n_at_maxit'])}")

## R4. Check the ratio fits: the band through the points

In [ ]:
%%only_for ratio
#======== R4. check the ratio fits:  the fitted band of every tsep must go through the points;  one page per pz, one panel per w ========
#points: the folded ratio of the fit (jackknife mean and error);  bands: the fitted curve of every tsep, jackknife mean and error over the samples;
#grey band: M00 of this w;  the pulls (data mean - mean curve) / error of every point are printed per w
check_model = "shared"                          #one of ratio_fit_models
check_gevp_time = (3, 5)                        #one of ratio_gevp_times
check_svdcut = 1e-2                             #one of ratio_fit_svdcuts
tsep_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948", "#8c564b", "#17becf", "#7f7f7f", "#bcbd22"]

t0, t_ref = check_gevp_time
pdf = PdfPages(os.path.join(save_dir, "plots", f"R4_ratio_fit_{check_model}_gevp{t0}-{t_ref}_svd{check_svdcut:g}_{run_tag}.pdf"))
for pz in ratio_fit_choice:
    d = np.load(os.path.join(save_dir, ratio_fit_name.format(model=check_model, t0=t0, t_ref=t_ref, pz=pz, svdcut=check_svdcut)))
    M = d["M_jk"]                                   # (jk:1504, w:10)
    Ai = d["Ai_jk"]                                 # (jk:1504, w:10) separate, (jk:1504, 1) shared
    Afi = d["Afi_jk"]                               # (jk:1504, w:10) separate, (jk:1504, 1) shared
    dE = d["dE_jk"]                                 # (jk:1504)
    n_w = M.shape[1]
    n_point_w = len(d["tsep"])                      #points of one w
    y_mean = d["y_mean"].reshape(n_w, n_point_w)    # (w:10, point_w:n)
    y_err = d["y_err"].reshape(n_w, n_point_w)      # (w:10, point_w:n)
    pull = d["pull"].reshape(n_w, n_point_w)        # (w:10, point_w:n)
    tsep_fit = sorted(set(d["tsep"].tolist()))      #the tsep of the window
    fig, axes = plt.subplots(2, 5, figsize=(16, 6.4))
    for iw in range(n_w):
        ax = axes.flat[iw]
        for i_tsep, tsep in enumerate(tsep_fit):
            color = tsep_colors[i_tsep % len(tsep_colors)]
            sel = d["tsep"] == tsep                                                     # (point_w:n)
            ax.errorbar(d["tau"][sel] - tsep / 2, y_mean[iw, sel], y_err[iw, sel], color=color, marker="o", ms=3, ls="none",
                        elinewidth=0.8, capsize=1.5, label=f"tsep {tsep}")
            tau_fine = np.linspace(d["tau"][sel].min(), tsep / 2, 30)                  # (tau_fine:30)
            e1 = np.exp(-dE[:, None] * tau_fine) + np.exp(-dE[:, None] * (tsep - tau_fine))     # (jk:1504, tau_fine:30)
            e2 = np.exp(-dE * tsep)[:, None]                                            # (jk:1504, 1)
            if check_model == "separate":
                curve = M[:, iw, None] + Ai[:, iw, None] * e1 + Afi[:, iw, None] * e2   # (jk:1504, tau_fine:30)
            else:
                curve = M[:, iw, None] * (1 + Ai[:, 0, None] * e1 + Afi[:, 0, None] * e2)
            c_mean, c_err = jk(curve)                                                   # (tau_fine:30), (tau_fine:30)
            ax.fill_between(tau_fine - tsep / 2, c_mean - c_err, c_mean + c_err, color=color, alpha=0.25, lw=0)
        M_mean, M_err = jk(M[:, iw])
        ax.axhspan(M_mean - M_err, M_mean + M_err, color="#898781", alpha=0.3, lw=0)
        ax.grid(color="#e1e0d9", lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"w = {iw}:  M00 = {gv.gvar(M_mean, M_err)},  mean pull {pull[iw].mean():+.2f}", fontsize=8)
        ax.tick_params(labelsize=7)
    axes.flat[0].legend(fontsize=6, frameon=False)
    for ax in axes[1]:
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$", fontsize=8)
    fig.suptitle(f"{plot_title}: ratio fit {check_model}, pz {pz}, window {ratio_fit_choice[pz]['window']}, GEVP t0 {t0}, t_ref {t_ref}, svdcut {check_svdcut:g};  "
                 f"chi2/dof {float(d['chi2']) / int(d['dof']):.3f}", fontsize=10)
    fig.tight_layout()
    save_figure(fig, pdf, "R4_ratio_fit", f"ratio_fit_{check_model}_pz{pz}")
    print(f"pz {pz}: mean pull per w " + " ".join(f"{p:+.2f}" for p in pull.mean(axis=1))
          + ";  rms pull per w " + " ".join(f"{p:.2f}" for p in np.sqrt((pull ** 2).mean(axis=1))))
pdf.close()

## R5. Matrix elements of the ratio fit: average over the GEVP times

In [ ]:
%%only_for ratio
#======== R5. matrix elements of the ratio fit:  M(pz, w) and dE, averaged over the GEVP times;  one file per model and svdcut ========
#in every jackknife sample: weight ~ exp(-chi2_j/2) over the (t0, t_ref) of ratio_gevp_times (FIT_PLAN.md section 9, D2):
#  every fit of a pz has the same window, so k and N_cut are the same for all and drop out;  with one GEVP time it is that fit
for model in ratio_fit_models:
    for svdcut in ratio_fit_svdcuts:
        M_all = []                      #M_all[pz]: (jk:1504, w:10)
        dE_all = []                     #dE_all[pz]: (jk:1504)
        weight_all = []                 #weight_all[pz]: (jk:1504, gevp_time:9)
        for pz in ratio_fit_choice:
            M_choice = []
            dE_choice = []
            chi2_choice = []
            k = []
            for t0, t_ref in ratio_gevp_times:
                file = os.path.join(save_dir, ratio_fit_name.format(model=model, t0=t0, t_ref=t_ref, pz=pz, svdcut=svdcut))
                assert os.path.exists(file), f"no fit {file}: run R3"
                d = np.load(file)
                assert str(d["settings"]) == ratio_settings_text(model, pz, svdcut), f"{file} was fitted with other settings than R0"
                M_choice.append(d["M_jk"])              # (jk:1504, w:10)
                dE_choice.append(d["dE_jk"])            # (jk:1504)
                chi2_choice.append(d["chi2_jk"])        # (jk:1504)
                k.append(int(d["k"]))
            M_choice = np.stack(M_choice, axis=1)       # (jk:1504, gevp_time:9, w:10)
            dE_choice = np.stack(dE_choice, axis=1)     # (jk:1504, gevp_time:9)
            chi2_choice = np.stack(chi2_choice, axis=1) # (jk:1504, gevp_time:9)
            M, dE, weight = aic_average(M_choice, dE_choice, chi2_choice, np.array(k), np.zeros(len(k)))
            M_all.append(M)
            dE_all.append(dE)
            weight_all.append(weight)
        M_all = np.stack(M_all, axis=2)                 # (jk:1504, w:10, pz:7)
        dE_all = np.stack(dE_all, axis=1)               # (jk:1504, pz:7)
        weight_all = np.stack(weight_all, axis=1)       # (jk:1504, pz:7, gevp_time:9)
        #---- save
        file = os.path.join(save_dir, ratio_result_name.format(model=model, svdcut=svdcut))
        if os.path.exists(file):
            print("replacing", file)
        with h5py.File(file, "w") as f:
            f["M_jk"] = M_all                           # (jk:1504, w:10, pz:7)
            f["dE_jk"] = dE_all                         # (jk:1504, pz:7)
            f["weight_jk"] = weight_all                 # (jk:1504, pz:7, gevp_time:9)
            f["gevp_times"] = np.array(ratio_gevp_times)                                        # (gevp_time:9, 2)
            f["settings"] = [ratio_settings_text(model, pz, svdcut) for pz in ratio_fit_choice]  # (pz:7)
            f["basis"] = [" ".join(ratio_basis[pz]) for pz in ratio_fit_choice]                # (pz:7)
            f["pz_list"] = list(ratio_fit_choice)
            with h5py.File(os.path.join(data_save_dir, ratio_data_name.format(t0=ratio_gevp_times[0][0], t_ref=ratio_gevp_times[0][1])), "r") as g:
                f["cfg_list"] = g["cfg_list"][()]       # (cfg:1504)  sample j leaves out cfg j
        #---- print
        M0_mean, M0_err = jk(M_all[:, 0])               # (pz:7), (pz:7)
        dE_mean, dE_err = jk(dE_all)                    # (pz:7), (pz:7)
        print(f"\nratio fit {model}, svdcut {svdcut:g}, average over {len(ratio_gevp_times)} GEVP times:")
        print("   M(w 0) pz 0-6: " + " ".join(str(gv.gvar(m, e)) for m, e in zip(M0_mean, M0_err)))
        print("   dE     pz 0-6: " + " ".join(str(gv.gvar(m, e)) for m, e in zip(dE_mean, dE_err)))
        print("   mean weight of each (t0, t_ref) " + " ".join(str(g) for g in ratio_gevp_times) + ":")
        for ipz, pz in enumerate(ratio_fit_choice):
            print(f"      pz {pz}: " + " ".join(f"{w:.2f}" for w in weight_all[:, ipz].mean(axis=0)))
        print("saved", file)

## S0. sGEVP fit: settings ($t_0$, windows, AIC grids, priors) and the model functions

In [ ]:
%%only_for sgevp
#======== S0. settings of the sGEVP fit, and its model functions ========
#every choice of part S is here, so that any section of part S can be run again after section 0 and this cell

#---- the GEVP times t0 of each pz (S2 makes M_eff at all of them;  S3 fits and S5 averages the t0 of each pz)
#pion forward pz 6 leaves out t0 6 (FIT_PLAN.md section 9, "Decision")
pion_forward_t0 = {0: [2, 3, 4, 5, 6], 1: [2, 3, 4, 5, 6], 2: [2, 3, 4, 5, 6], 3: [2, 3, 4, 5, 6], 4: [2, 3, 4, 5, 6], 5: [2, 3, 4, 5, 6], 6: [2, 3, 4, 5]}
proton_forward_t0 = {0: [2, 3, 4, 5, 6], 1: [2, 3, 4, 5, 6], 2: [2, 3, 4, 5, 6], 3: [2, 3, 4, 5, 6], 4: [2, 3, 4, 5, 6], 5: [2, 3, 4, 5, 6], 6: [2, 3, 4, 5, 6]}
pion_sym_t0 = proton_forward_t0                 #off-forward: no t0 left out (FIT_PLAN.md section 15)
proton_sym_t0 = proton_forward_t0
#the end of the signal of each pz (S2):  the last t with |M_eff(w 0)| / error >= signal_sn_min at every t from signal_from_t on, at t0 signal_t0, at most signal_t_cap
signal_t0 = 2
signal_from_t = 2
signal_sn_min = 2.0
signal_t_cap = 16

#---- the fits (S3)
sgevp_svdcuts = [1e-2]
#(first t, last t) of each pz;  t = source-sink separation, M_eff(t) = R(t+1) - R(t)
sgevp_window_fixed = {0: (2, 12), 1: (2, 12), 2: (2, 9), 3: (2, 8), 4: (2, 7), 5: (2, 6), 6: (2, 6)}
sgevp_windows = "fixed"                         #"fixed": the fixed window only;  "B1": every window of the B1 grid (it holds the fixed window and the B2 grid)
#the AIC grids of each pz (FIT_PLAN.md section 7):  {pz: {first t: (smallest last t, largest last t)}};  every last t in between is one window
#  B1 = first t 2, 3, 4, last t from first t + 4 to the end of the signal;  B2 = first t 2 only;  the ends are those of the 1504-cfg data (S2)
pion_forward_aic_B1 = {0: {2: (6, 16), 3: (7, 16), 4: (8, 16)}, 1: {2: (6, 16), 3: (7, 16), 4: (8, 16)}, 2: {2: (6, 16), 3: (7, 16), 4: (8, 16)},
                       3: {2: (6, 9), 3: (7, 9), 4: (8, 9)}, 4: {2: (6, 8), 3: (7, 8), 4: (8, 8)}, 5: {2: (6, 6)}, 6: {2: (6, 6)}}
pion_forward_aic_B2 = {0: {2: (6, 16)}, 1: {2: (6, 16)}, 2: {2: (6, 16)}, 3: {2: (6, 9)}, 4: {2: (6, 8)}, 5: {2: (6, 6)}, 6: {2: (6, 6)}}
proton_forward_aic_B1 = {0: {2: (6, 16), 3: (7, 16), 4: (8, 16)}, 1: {2: (6, 15), 3: (7, 15), 4: (8, 15)}, 2: {2: (6, 12), 3: (7, 12), 4: (8, 12)},
                         3: {2: (6, 11), 3: (7, 11), 4: (8, 11)}, 4: {2: (6, 11), 3: (7, 11), 4: (8, 11)}, 5: {2: (6, 9), 3: (7, 9), 4: (8, 9)},
                         6: {2: (6, 8), 3: (7, 8), 4: (8, 8)}}
proton_forward_aic_B2 = {0: {2: (6, 16)}, 1: {2: (6, 15)}, 2: {2: (6, 12)}, 3: {2: (6, 11)}, 4: {2: (6, 11)}, 5: {2: (6, 9)}, 6: {2: (6, 8)}}
pion_sym_aic_B1 = {0: {2: (6, 16), 3: (7, 16), 4: (8, 16)}, 1: {2: (6, 16), 3: (7, 16), 4: (8, 16)}, 2: {2: (6, 14), 3: (7, 14), 4: (8, 14)},
                   3: {2: (6, 10), 3: (7, 10), 4: (8, 10)}, 4: {2: (6, 9), 3: (7, 9), 4: (8, 9)}, 5: {2: (6, 7), 3: (7, 7)}, 6: {2: (6, 6)}}
pion_sym_aic_B2 = {0: {2: (6, 16)}, 1: {2: (6, 16)}, 2: {2: (6, 14)}, 3: {2: (6, 10)}, 4: {2: (6, 9)}, 5: {2: (6, 7)}, 6: {2: (6, 6)}}
proton_sym_aic_B1 = {0: {2: (6, 13), 3: (7, 13), 4: (8, 13)}, 1: {2: (6, 14), 3: (7, 14), 4: (8, 14)}, 2: {2: (6, 13), 3: (7, 13), 4: (8, 13)},
                     3: {2: (6, 11), 3: (7, 11), 4: (8, 11)}, 4: {2: (6, 10), 3: (7, 10), 4: (8, 10)}, 5: {2: (6, 10), 3: (7, 10), 4: (8, 10)},
                     6: {2: (6, 9), 3: (7, 9), 4: (8, 9)}}
proton_sym_aic_B2 = {0: {2: (6, 13)}, 1: {2: (6, 14)}, 2: {2: (6, 13)}, 3: {2: (6, 11)}, 4: {2: (6, 10)}, 5: {2: (6, 10)}, 6: {2: (6, 9)}}
#model M_eff,w(t) = A_w + B_w t e^(-dE t):  all ten w in one fit with one gap;  A_w = M(pz, w) < 0 (the fit varies log(-A))
sgevp_minus_A_prior = (1.0, 10.0)               #(center, width) of -A:  log(-A) = log(1(10)) = 0(10)
sgevp_B_prior = (0.0, 10.0)
sgevp_dE_prior = (0.5, 0.5)                     #log-normal:  the fit varies log(dE)
#the fit of the mean keeps the lowest chi2 of six starts:  1-5 one per dE of sgevp_dE_starts (-A = -M_eff at the last t, at least sgevp_minus_A_min;
#B from the first point);  6 = the fit with A free in sign (prior sgevp_free_A_prior) from starts 1-5, the lowest chi2, every A above -sgevp_minus_A_min set to it
sgevp_dE_starts = [0.15, 0.3, 0.5, 0.8, 1.2]
sgevp_free_A_prior = (0.0, 10.0)
sgevp_minus_A_min = 1e-3
sgevp_maxit = 20000
sgevp_tol = 1e-8
sgevp_fitter = "scipy_least_squares"
#fixed:  the six operators at every pz;  full covariance of all t and w;  every jackknife sample with its own covariance
sgevp_settings_text = (f"A < 0, -A {sgevp_minus_A_prior}, B {sgevp_B_prior}, dE {sgevp_dE_prior}, starts {sgevp_dE_starts} + sign-free start "
                       f"A {sgevp_free_A_prior}, -A at least {sgevp_minus_A_min}, maxit {sgevp_maxit}, tol {sgevp_tol}, {sgevp_fitter}, covariance per sample")

#---- the dictionaries of this hadron and frame, and the windows of each pz
if hadron == "pion" and frame == "forward":
    sgevp_t0, aic_B1, aic_B2 = pion_forward_t0, pion_forward_aic_B1, pion_forward_aic_B2
elif hadron == "pion":
    sgevp_t0, aic_B1, aic_B2 = pion_sym_t0, pion_sym_aic_B1, pion_sym_aic_B2
elif frame == "forward":
    sgevp_t0, aic_B1, aic_B2 = proton_forward_t0, proton_forward_aic_B1, proton_forward_aic_B2
else:
    sgevp_t0, aic_B1, aic_B2 = proton_sym_t0, proton_sym_aic_B1, proton_sym_aic_B2
sgevp_t0_list = sorted(set(t0 for pz in sgevp_t0 for t0 in sgevp_t0[pz]))     #every t0 that S2 makes:  [2, 3, 4, 5, 6]
assert sgevp_windows in ["fixed", "B1"], f"sgevp_windows = {sgevp_windows}: use fixed or B1"
sgevp_window_list = {}          #sgevp_window_list[pz][variant]: the windows (first t, last t) of "fixed", "B1", "B2"
for pz in sgevp_window_fixed:
    sgevp_window_list[pz] = {"fixed": [sgevp_window_fixed[pz]], "B1": [], "B2": []}
    for first in aic_B1[pz]:
        for last in range(aic_B1[pz][first][0], aic_B1[pz][first][1] + 1):
            sgevp_window_list[pz]["B1"].append((first, last))
    for first in aic_B2[pz]:
        for last in range(aic_B2[pz][first][0], aic_B2[pz][first][1] + 1):
            sgevp_window_list[pz]["B2"].append((first, last))
    assert sgevp_window_fixed[pz] in sgevp_window_list[pz]["B1"], f"pz {pz}: the fixed window is not in the B1 grid"
    assert set(sgevp_window_list[pz]["B2"]) <= set(sgevp_window_list[pz]["B1"]), f"pz {pz}: the B2 grid is not in the B1 grid"
if sgevp_windows == "fixed":
    sgevp_variants = ["fixed"]                  #S5 can average only what S3 fitted
else:
    sgevp_variants = ["fixed", "B1", "B2"]


def meff_model(x, p):
    #M_eff,w(t) = A_w + B_w t e^(-dE t);  x["iw"] = the w index of every point
    return p["A"][x["iw"]] + p["B"][x["iw"]] * x["t"] * gv.exp(-p["dE"][0] * x["t"])


def meff_prior(n_w):
    #log(-A_w) = 0(10), B_w, log(dE)
    p = gv.BufferDict()
    p["neglog(A)"] = gv.log(gv.gvar(np.full(n_w, sgevp_minus_A_prior[0]), np.full(n_w, sgevp_minus_A_prior[1])))
    p["B"] = gv.gvar(np.full(n_w, sgevp_B_prior[0]), np.full(n_w, sgevp_B_prior[1]))
    p["log(dE)"] = gv.log(gv.gvar([float(sgevp_dE_prior[0])], [float(sgevp_dE_prior[1])]))
    return p


def meff_prior_free(n_w):
    #A free in sign:  only for the sixth start
    p = gv.BufferDict()
    p["A"] = gv.gvar(np.full(n_w, sgevp_free_A_prior[0]), np.full(n_w, sgevp_free_A_prior[1]))
    p["B"] = gv.gvar(np.full(n_w, sgevp_B_prior[0]), np.full(n_w, sgevp_B_prior[1]))
    p["log(dE)"] = gv.log(gv.gvar([float(sgevp_dE_prior[0])], [float(sgevp_dE_prior[1])]))
    return p


print(f"{hadron_tag}, {frame}:  t0 made in S2 {sgevp_t0_list};  windows fitted: {sgevp_windows};  svdcuts {sgevp_svdcuts}")
for pz in sgevp_window_fixed:
    print(f"pz {pz}: t0 {sgevp_t0[pz]}, fixed window {sgevp_window_fixed[pz]}, B1 {len(sgevp_window_list[pz]['B1'])} windows "
          f"(last t up to {max(w[1] for w in sgevp_window_list[pz]['B1'])}), B2 {len(sgevp_window_list[pz]['B2'])} windows")

## S1. sGEVP data: the 2pt matrix $C(t)$ and the summed 3pt of the six operators

In [ ]:
%%only_for sgevp
#======== S1. sGEVP data:  the 2pt matrix C(t) and the summed 3pt pt3_summed(t) of the six operators;  saved in two files ========
#(section 1 of matrix_element_analysis_1504cfg.ipynb;  for sym the same per q, then the average over the four q, as part I of FIT_PLAN.md)
#pt3_summed_ab(t) = sum over tau = 1 .. t-1 of the method-1 3pt:  per source, C3input_ab(t) times O3 summed over the slices strictly between source and sink,
#  mean over the sources, minus jackknife(C3input_ab, mean over the sources) x jackknife(the summed O3, mean over the sources);  times c_sign
#C_ab(t):  the 2pt matrix, mean over the sources, times c_sign, jackknife, Hermitian part;  forward: of the C3input pack;  sym: of the C2pf pack
#a = sink operator, b = source operator;  the two halves are kept apart (the GEVP of S2 is solved in each half)
time_start = time.time()
half_sign = {"forward": +1, "backward": -1}     #the operator sits at t_src + tau (forward half) or t_src - tau (backward half)

#---- O3 of every cfg and q (the same lines as in R1)
O3 = {}                         #O3[q]: (cfg:1504, w:10, t:96)
for q in q_list:
    O3_parts = []
    for data in data_sets:
        file_2pt = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input",
                                                             ncfg=data["ncfg"], hadron=hadron_tag))
        with h5py.File(file_2pt, "r") as f:
            cfg_d = f["cfg_list"][()]                                       # (cfg:724|781)
        cfg_d = cfg_d[~np.isin(cfg_d, data["skip_cfg"])]                    # (cfg:724|780)  the 2pt cfgs of this data set
        file_ff = os.path.join(data["dir"], ff_name.format(qx=q[0], qy=q[1], qz=q[2], ncfg=data["ff_ncfg"]))
        print("reading", file_ff)
        with h5py.File(file_ff, "r") as f:
            assert tuple(f["q"][()]) == q, f"{file_ff}: another q"
            cfg_ff = f["cfg_list"][()]                                      # (cfg:800|798)
            idx = np.searchsorted(cfg_ff, cfg_d)                            # (cfg:724|780)  where the 2pt cfgs are in the FF file
            assert np.array_equal(cfg_ff[idx], cfg_d), f"{data['name']}: some 2pt cfgs are not in the FF file"
            ihyp = list(f["hyp_list"][()]).index(hyp)
            w_list = f["w_list"][()]                                        # (w:10)
            TXTX = f["FF"][idx, 3, 3, ihyp]                                 # (cfg:724|780, w:10, t:96)
            TYTY = f["FF"][idx, 4, 4, ihyp]                                 # (cfg:724|780, w:10, t:96)
            XYXY = f["FF"][idx, 0, 0, ihyp]                                 # (cfg:724|780, w:10, t:96)
        O3_parts.append(TXTX + TYTY - 2 * XYXY)
    O3[q] = np.concatenate(O3_parts)                                        # (cfg:1504, w:10, t:96)

#---- C and pt3_summed of each half:  every q, then the average over the q
C_jk = {}                       #C_jk[half]: (jk:1504, pz:7, t:20, a:6, b:6)
pt3_summed_jk = {}              #pt3_summed_jk[half]: (jk:1504, w:10, pz:7, t:20, a:6, b:6)
for half in half_sign:
    C_cfg_sum = 0               #becomes (cfg:1504, a:6, b:6, pz:7, t:20):  the 2pt of the GEVP, summed over q / number of q
    pt3_summed_sum = 0          #becomes (jk:1504, w:10, pz:7, t:20, a:6, b:6)
    for q in q_list:
        #read the C3input pack of q, every source;  in the sym frame also the C2pf pack of q (the 2pt of the GEVP)
        pt2_parts = []
        c2pf_parts = []
        t_src_parts = []
        cfg_parts = []
        for data in data_sets:
            file_2pt = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input",
                                                                 ncfg=data["ncfg"], hadron=hadron_tag))
            print("reading", file_2pt, half)
            with h5py.File(file_2pt, "r") as f:
                cfg_d = f["cfg_list"][()]                                   # (cfg:724|781)
                pz_list = f["pz_list"][()]                                  # (pz:7)
                # pt2_d: (cfg:724|781, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, t_src:24, pz:7, t:20)
                pt2_d = f[f"pt2_{half}"][()]
                t_src_d = f["t_src_list"][()]                               # (cfg:724|781, t_src:24)  the slice of every source
            keep = ~np.isin(cfg_d, data["skip_cfg"])                        # (cfg:724|781)
            pt2_parts.append(pt2_d[keep])
            t_src_parts.append(t_src_d[keep])
            cfg_parts.append(cfg_d[keep] + data["cfg_shift"])
            del pt2_d
            if frame == "sym":
                file_c2pf = os.path.join(data["dir"], pt2_name.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C2pf",
                                                                      ncfg=data["ncfg"], hadron=hadron_tag))
                with h5py.File(file_c2pf, "r") as f:
                    assert np.array_equal(f["cfg_list"][()], cfg_d), f"{file_c2pf}: other cfgs than the C3input pack"
                    # c2pf_d: (cfg:724|781, smear_snk:2, smear_src:2, gamma_snk:3, gamma_src:3, t_src:24, pz:7, t:20)
                    c2pf_d = f[f"pt2_{half}"][()]
                c2pf_parts.append(c2pf_d[keep])
                del c2pf_d
        cfg_list = np.concatenate(cfg_parts)                                # (cfg:1504)
        assert len(cfg_list) == ncfg and len(np.unique(cfg_list)) == ncfg, "the cfg numbers of the data sets overlap, or skip_cfg is wrong"
        t_src_list = np.concatenate(t_src_parts)                            # (cfg:1504, t_src:24)
        n_cfg, n_tsrc = t_src_list.shape
        Cij = op_matrix(np.concatenate(pt2_parts), op_labels)               # (cfg:1504, a:6, b:6, t_src:24, pz:7, t:20)
        del pt2_parts
        n_tsep = Cij.shape[-1]
        C3_cfg = Cij.mean(axis=3)                                           # (cfg:1504, a:6, b:6, pz:7, t:20)  mean over the sources
        #the 2pt of the GEVP
        if frame == "forward":
            C_cfg_sum = C_cfg_sum + C3_cfg / len(q_list)
        else:
            C_cfg_sum = C_cfg_sum + op_matrix(np.concatenate(c2pf_parts), op_labels).mean(axis=3) / len(q_list)
            del c2pf_parts
        #O3 summed over tau = 1 .. t-1 on the slices of every source
        i_cfg = np.arange(n_cfg)[:, None, None]
        t_op = (t_src_list[:, :, None] + half_sign[half] * np.arange(n_tsep)) % Lt     # (cfg:1504, t_src:24, tau:20)
        O_tau = O3[q][i_cfg, :, t_op]                                       # (cfg:1504, t_src:24, tau:20, w:10)  numpy puts w last here
        O_sum = np.zeros_like(O_tau)                                        # (cfg:1504, t_src:24, t:20, w:10)
        for t in range(n_tsep):
            O_sum[:, :, t] = O_tau[:, :, 1:t].sum(axis=2)
        #the 3pt of method 1, summed over tau
        connected = np.einsum("cabspt,cstw->cabwpt", Cij, O_sum, optimize=True) / n_tsrc                   # (cfg:1504, a:6, b:6, w:10, pz:7, t:20)
        vacuum = np.einsum("cabpt,ctw->cabwpt", jackknife(C3_cfg), jackknife(O_sum.mean(axis=1)), optimize=True)    # (jk:1504, a:6, b:6, w:10, pz:7, t:20)
        pt3_summed_q = np.moveaxis(c_sign * (jackknife(connected) - vacuum), (1, 2), (-2, -1))             # (jk:1504, w:10, pz:7, t:20, a:6, b:6)
        pt3_summed_sum = pt3_summed_sum + pt3_summed_q / len(q_list)
        del Cij, C3_cfg, O_tau, O_sum, connected, vacuum, pt3_summed_q
        print(f"  {half}, q {q}: done, {time.time() - time_start:.0f} s")
    C = np.moveaxis(c_sign * jackknife(C_cfg_sum), (1, 2), (-2, -1))       # (jk:1504, pz:7, t:20, a:6, b:6)
    C_jk[half] = 0.5 * (C + np.conj(np.swapaxes(C, -1, -2)))               #Hermitian part
    pt3_summed_jk[half] = pt3_summed_sum
    del C_cfg_sum, pt3_summed_sum, C

#---- save the two files:  everything the sGEVP needs (the files to give to collaborators)
file = os.path.join(data_save_dir, pt2_6x6_name)
if os.path.exists(file):
    print("replacing", file)
with h5py.File(file, "w") as f:
    for half in half_sign:
        f[f"pt2_jk_{half}"] = C_jk[half]                # (jk:1504, pz:7, t:20, a:6, b:6)  c_sign in, Hermitian part
    f["cfg_list"] = cfg_list                            # (cfg:1504)  jackknife sample j leaves out cfg j
    f["pz_list"] = pz_list
    f["op_labels"] = op_labels                          #the order of a and b
    f["q_list"] = q_list                                #the q averaged over
    f["c_sign"] = c_sign
    f["hadron_tag"] = hadron_tag
    f["data_sets"] = [data["name"] for data in data_sets]
    f["data_set_skip_cfg"] = [" ".join(str(c) for c in data["skip_cfg"]) for data in data_sets]
print("saved", file)
file = os.path.join(data_save_dir, pt3_summed_6x6_name)
if os.path.exists(file):
    print("replacing", file)
with h5py.File(file, "w") as f:
    for half in half_sign:
        f[f"pt3_summed_jk_{half}"] = pt3_summed_jk[half]    # (jk:1504, w:10, pz:7, t:20, a:6, b:6)  c_sign in
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["op_labels"] = op_labels
    f["q_list"] = q_list
    f["c_sign"] = c_sign
    f["hyp"] = hyp
    f["hadron_tag"] = hadron_tag
    f["data_sets"] = [data["name"] for data in data_sets]
    f["data_set_skip_cfg"] = [" ".join(str(c) for c in data["skip_cfg"]) for data in data_sets]
print("saved", file)

#check:  the summed ratio pt3_summed(t 8) / C(t 8) of BG5 at w 0 (one operator, before any GEVP)
for half in half_sign:
    mean, err = jk(pt3_summed_jk[half][:, 0, :, 8, 0, 0].real / C_jk[half][:, :, 8, 0, 0].real)       # (pz:7), (pz:7)
    print(f"{half}, BG5: pt3_summed(8)/C(8) at w 0, pz 0-6: " + " ".join(f"{m:+.3f}({e * 1e3:.0f})" for m, e in zip(mean, err)))
print(f"{time.time() - time_start:.0f} s")

## S2. sGEVP: $M_{\rm eff}(t)$ at every $t_0$, the end of the signal, plots

In [ ]:
%%only_for sgevp
#======== S2. sGEVP:  R(t) = u^dag pt3_summed(t) u / lambda - u^dag pt3_summed(t0) u,  M_eff(t) = R(t+1) - R(t),  at every t0;  saved, with plots ========
#u, lambda:  the ground state of C(t) u = lambda C(t0) u, solved again at every t and in every jackknife sample, with u^dag C(t0) u = 1
#  t > t0:  the largest lambda (= e^(-E0 (t - t0)) < 1);   t < t0:  the smallest lambda (= e^(E0 (t0 - t)) > 1);   R(t0) = 0
#each half has its own GEVP;  R is the mean of the two halves;  Re[u^dag pt3_summed u] keeps the Hermitian part of pt3_summed
#the end of the signal of each pz:  S0 (signal_*);  the AIC grids of S0 end there
meff_plot_err_max = 0.1         #plots: points with a larger error are left out
meff_plot_t_after_end = 2       #plots: t up to this many steps after the end of the signal
t0_colors = {2: "#2a78d6", 3: "#eb6834", 4: "#1baf7a", 5: "#e87ba4", 6: "#6250d6"}
time_start = time.time()

#---- the two files of S1
C_jk = {}
pt3_summed_jk = {}
with h5py.File(os.path.join(data_save_dir, pt2_6x6_name), "r") as f:
    for half in ["forward", "backward"]:
        C_jk[half] = f[f"pt2_jk_{half}"][()]                           # (jk:1504, pz:7, t:20, a:6, b:6)
    cfg_list = f["cfg_list"][()]                                        # (cfg:1504)
    pz_list = f["pz_list"][()]                                          # (pz:7)
with h5py.File(os.path.join(data_save_dir, pt3_summed_6x6_name), "r") as f:
    for half in ["forward", "backward"]:
        pt3_summed_jk[half] = f[f"pt3_summed_jk_{half}"][()]           # (jk:1504, w:10, pz:7, t:20, a:6, b:6)
    w_list = f["w_list"][()]                                            # (w:10)
n_jk, n_pz, n_tsep = C_jk["forward"].shape[:3]
n_w = len(w_list)

#---- M_eff at every t0
meff_jk = {}                    #meff_jk[t0]: (jk:1504, pz:7, w:10, t:20), NaN where R(t+1) or R(t) is not made
for t0 in sgevp_t0_list:
    meff_jk[t0] = np.full((n_jk, n_pz, n_w, n_tsep), np.nan)
    for ipz in range(n_pz):
        R = np.zeros((n_jk, n_w, n_tsep))                               # (jk:1504, w:10, t:20)  the mean of the two halves
        for half in ["forward", "backward"]:
            C = C_jk[half][:, ipz]                                      # (jk:1504, t:20, a:6, b:6)
            K = pt3_summed_jk[half][:, :, ipz]                          # (jk:1504, w:10, t:20, a:6, b:6)
            R_half = np.full((n_jk, n_w, n_tsep), np.nan)               # (jk:1504, w:10, t:20)
            R_half[:, :, t0] = 0
            for t in range(1, n_tsep):
                if t == t0:
                    continue
                lam, u_all = gevp(C[:, t], C[:, t0])                    # (jk:1504, state:6), (jk:1504, a:6, state:6)
                if t > t0:
                    k = -1                                              #the largest lambda
                else:
                    k = 0                                               #the smallest lambda
                u = u_all[:, :, k]                                      # (jk:1504, a:6)
                u_K_u = np.einsum("ja,jwab,jb->jw", u.conj(), K[:, :, t], u).real          # (jk:1504, w:10)
                u_K0_u = np.einsum("ja,jwab,jb->jw", u.conj(), K[:, :, t0], u).real        # (jk:1504, w:10)
                R_half[:, :, t] = u_K_u / lam[:, k][:, None] - u_K0_u
            R = R + 0.5 * R_half
        meff_jk[t0][:, ipz, :, :-1] = R[:, :, 1:] - R[:, :, :-1]
del C_jk, pt3_summed_jk

#---- the end of the signal of each pz
meff_mean, meff_err = jk(meff_jk[signal_t0])                            # (pz:7, w:10, t:20), (pz:7, w:10, t:20)
signal_end = np.zeros(n_pz, dtype=int)                                  # (pz:7)
for ipz in range(n_pz):
    end = signal_from_t
    while end + 1 <= signal_t_cap and abs(meff_mean[ipz, 0, end + 1]) >= signal_sn_min * meff_err[ipz, 0, end + 1]:
        end = end + 1
    signal_end[ipz] = end
print(f"M_eff(t) at w 0, t0 {signal_t0}:")
print(f"{'t':>3s} " + " ".join(f"{'pz ' + str(pz):>13s}" for pz in pz_list))
for t in range(1, 15):
    print(f"{t:3d} " + " ".join(f"{str(gv.gvar(meff_mean[i, 0, t], meff_err[i, 0, t])) if np.isfinite(meff_mean[i, 0, t]) else '-':>13s}" for i in range(n_pz)))
print(f"end of the signal, pz 0-{n_pz - 1}: " + " ".join(str(e) for e in signal_end) + "   (the AIC grids of S0 run up to these t)")

#---- save
file = os.path.join(data_save_dir, meff_name)
if os.path.exists(file):
    print("replacing", file)
with h5py.File(file, "w") as f:
    for t0 in sgevp_t0_list:
        f[f"meff_jk_t0{t0}"] = meff_jk[t0]                              # (jk:1504, pz:7, w:10, t:20)
    f["t0_list"] = sgevp_t0_list
    f["signal_end"] = signal_end                                        # (pz:7)
    f["signal_rule"] = f"|M_eff(w 0)| / error >= {signal_sn_min} at every t from {signal_from_t} on, t0 {signal_t0}, at most t {signal_t_cap}"
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["op_labels"] = op_labels
    f["hadron_tag"] = hadron_tag
print("saved", file, f";  {time.time() - time_start:.0f} s")

#---- plots:  one page per pz, one panel per w, M_eff(t) of every t0 of this pz
pdf = PdfPages(os.path.join(save_dir, "plots", f"S2_sgevp_meff_{run_tag}.pdf"))
for ipz, pz in enumerate(pz_list):
    t_max = min(signal_end[ipz] + meff_plot_t_after_end, n_tsep - 2)
    fig, axes = plt.subplots(2, 5, figsize=(16, 6.4), sharex=True)
    for iw in range(n_w):
        ax = axes.flat[iw]
        for i_t0, t0 in enumerate(sgevp_t0[int(pz)]):
            m, e = jk(meff_jk[t0][:, ipz, iw])                          # (t:20), (t:20)
            t = np.arange(1, t_max + 1)
            ok = np.isfinite(m[t]) & (e[t] < meff_plot_err_max)
            ax.errorbar(t[ok] + 0.08 * (i_t0 - 2), m[t][ok], e[t][ok], color=t0_colors[t0], marker="o", ms=3, ls="none",
                        elinewidth=0.8, capsize=1.5, label=f"t0 {t0}")
        ax.axvline(signal_end[ipz] + 0.5, color="#898781", ls=":", lw=0.8)
        ax.axhline(0, color="#c3c2b7", lw=0.5)
        ax.grid(color="#e1e0d9", lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"w = {iw}", fontsize=9)
        ax.tick_params(labelsize=7)
    axes.flat[0].legend(fontsize=6, frameon=False)
    for ax in axes[1]:
        ax.set_xlabel("t", fontsize=8)
    fig.suptitle(f"{plot_title}: sGEVP M_eff(t), pz {pz};  dotted: end of the signal (t {signal_end[ipz]});  "
                 f"points with error > {meff_plot_err_max} left out", fontsize=10)
    fig.tight_layout()
    save_figure(fig, pdf, "S2_meff", f"sgevp_meff_pz{pz}")
pdf.close()

## S3. sGEVP fits $M_{\rm eff}(t) = A_w + B_w\, t\, e^{-\Delta E\, t}$: every pz, $t_0$, window and svdcut

In [ ]:
%%only_for sgevp
#======== S3. sGEVP fits:  every pz, t0 of the pz, window and svdcut of S0;  one file per fit;  a fit whose file is saved with the same settings is not done again ========
#the model, the windows and the priors are in S0;  the fit itself is fit_jackknife (section 0)


def sgevp_fit_one(pz, t0, svdcut, window):
    #one sGEVP fit in every jackknife sample;  saves its file and returns one line of text
    time_fit = time.time()
    first, last = window
    t = np.arange(first, last + 1)                                          # (t:n)  the t of the window
    with h5py.File(os.path.join(data_save_dir, meff_name), "r") as f:
        ipz = list(f["pz_list"][()]).index(pz)
        meff = f[f"meff_jk_t0{t0}"][:, ipz]                                # (jk:1504, w:10, t:20)
    y = meff[:, :, t]                                                       # (jk:1504, w:10, t:n)
    assert not np.isnan(y).any(), f"pz {pz}, t0 {t0}: NaN in M_eff inside the window {window}"
    n_jk, n_w = y.shape[:2]
    y_jk = y.reshape(n_jk, n_w * len(t))                                    # (jk:1504, point:10 n)  all w after each other
    x = {"t": np.tile(t.astype(float), n_w), "iw": np.repeat(np.arange(n_w), len(t))}     #the t and w index of every point
    #---- starts 1-5:  one per dE of sgevp_dE_starts;  -A = -M_eff at the last t (at least sgevp_minus_A_min), B from the first point
    y_mean_w = y.mean(axis=0)                                               # (w:10, t:n)
    starts = []
    for dE_start in sgevp_dE_starts:
        starts.append({"neglog(A)": np.log(np.maximum(-y_mean_w[:, -1], sgevp_minus_A_min)),
                       "B": (y_mean_w[:, 0] - y_mean_w[:, -1]) / (t[0] * np.exp(-dE_start * t[0])),
                       "log(dE)": np.log([dE_start])})
    #---- start 6:  the fit of the mean with A free in sign, from starts 1-5;  the lowest chi2;  every A above -sgevp_minus_A_min set to -sgevp_minus_A_min
    y_mean = y_jk.mean(axis=0)                                              # (point:10 n)
    d = y_jk - y_mean                                                       # (jk:1504, point:10 n)
    cov_all = (n_jk - 1) / n_jk * (d.T @ d)                                 # (point:10 n, point:10 n)
    free = None
    chi2_free = np.nan
    for p0 in starts:
        gv.switch_gvar()
        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_mean, cov_all)), fcn=meff_model, prior=meff_prior_free(n_w),
                                   p0={"A": y_mean_w[:, -1], "B": p0["B"], "log(dE)": p0["log(dE)"]},
                                   svdcut=svdcut, maxit=sgevp_maxit, tol=sgevp_tol, fitter=sgevp_fitter)
        if free is None or fit.chi2 < chi2_free:
            chi2_free = float(fit.chi2)
            free = {"A": np.array(fit.pmean["A"]), "B": np.array(fit.pmean["B"]), "log(dE)": np.array(fit.pmean["log(dE)"])}
        gv.restore_gvar()
    starts.append({"neglog(A)": np.log(np.maximum(-free["A"], sgevp_minus_A_min)), "B": free["B"], "log(dE)": free["log(dE)"]})
    #---- the fit in every jackknife sample
    result = fit_jackknife(y_jk, x, meff_model, meff_prior, (n_w,), starts, ["A", "B", "dE"],
                           svdcut, sgevp_maxit, sgevp_tol, sgevp_fitter)
    file = os.path.join(save_dir, sgevp_fit_name.format(t0=t0, pz=pz, svdcut=svdcut, first=first, last=last))
    np.savez(file, M_jk=result["param_jk"]["A"], B_jk=result["param_jk"]["B"], dE_jk=result["param_jk"]["dE"][:, 0],
             chi2_jk=result["chi2_jk"], chi2=result["chi2"], dof=result["dof"], svdn=result["svdn"], n_at_maxit=result["n_at_maxit"],
             chi2_starts=result["chi2_starts"], chi2_free=chi2_free, y_mean=result["y_mean"], y_err=result["y_err"],
             curve_mean=result["curve_mean"], pull=result["pull"], t=t, k=2 * n_w + 1, n_point=len(y_mean),
             window=np.array(window), t0=t0, svdcut=svdcut, n_jk=n_jk, settings=sgevp_settings_text)
    return f"{os.path.basename(file)}: chi2/dof {result['chi2'] / result['dof']:.3f}, {result['n_at_maxit']} fits at maxit, {time.time() - time_fit:.0f} s"


#---- the fits that are not saved yet
assert os.path.exists(os.path.join(data_save_dir, meff_name)), "no M_eff file: run S2"
tasks = []                      #(pz, t0, svdcut, window)
n_all = 0
for pz in sgevp_window_fixed:
    for t0 in sgevp_t0[pz]:
        os.makedirs(os.path.join(save_dir, "sgevp_fits", f"t0{t0}"), exist_ok=True)
        for svdcut in sgevp_svdcuts:
            for first, last in sgevp_window_list[pz][sgevp_windows]:
                n_all = n_all + 1
                file = os.path.join(save_dir, sgevp_fit_name.format(t0=t0, pz=pz, svdcut=svdcut, first=first, last=last))
                if os.path.exists(file):
                    saved = str(np.load(file)["settings"])
                    assert saved == sgevp_settings_text, (f"{file} was fitted with other settings:\n  {saved}\n"
                                                         f"delete the file to fit it again with the settings of S0")
                else:
                    tasks.append((pz, t0, svdcut, (first, last)))


def window_length(task):
    #the number of t in the window of a task:  the longest windows are fitted first, so that the workers finish at about the same time
    return task[3][1] - task[3][0]


tasks.sort(key=window_length, reverse=True)
print(f"{n_all} fits in all, {n_all - len(tasks)} already saved, {len(tasks)} to do with {n_workers} workers", flush=True)
time_start = time.time()
lines = Parallel(n_jobs=n_workers, verbose=5)(delayed(sgevp_fit_one)(*task) for task in tasks)
print(f"done, {time.time() - time_start:.0f} s")

#---- a table of the fixed-window fits:  chi2/dof, M(w 0), dE, fits that stopped at maxit
for svdcut in sgevp_svdcuts:
    print(f"\nsvdcut {svdcut:g}, fixed windows:   pz  t0  window   chi2/dof   M(w 0)          dE              at maxit")
    for pz in sgevp_window_fixed:
        first, last = sgevp_window_fixed[pz]
        for t0 in sgevp_t0[pz]:
            d = np.load(os.path.join(save_dir, sgevp_fit_name.format(t0=t0, pz=pz, svdcut=svdcut, first=first, last=last)))
            M0_mean, M0_err = jk(d["M_jk"][:, 0])
            dE_mean, dE_err = jk(d["dE_jk"])
            print(f"{'':32s}{pz}   {t0}   {first}-{last:<4d}   {float(d['chi2']) / int(d['dof']):6.3f}    {str(gv.gvar(M0_mean, M0_err)):14s}  "
                  f"{str(gv.gvar(dE_mean, dE_err)):14s}  {int(d['n_at_maxit'])}")

## S4. Check the sGEVP fits: the band through the points

In [ ]:
%%only_for sgevp
#======== S4. check the sGEVP fits:  the fitted band must go through the points;  one page per pz and t0 (fixed window), one panel per w ========
#points: M_eff(t), filled inside the window, open outside;  band: the fitted curve, jackknife mean and error over the samples;  grey band: M of this w
check_svdcut = 1e-2                             #one of sgevp_svdcuts
check_t_max = 14                                #points up to this t
check_err_max = 0.3                             #points with a larger error are left out

with h5py.File(os.path.join(data_save_dir, meff_name), "r") as f:
    pz_list = list(f["pz_list"][()])
    meff_mean = {}
    meff_err = {}
    for t0 in sgevp_t0_list:
        meff_mean[t0], meff_err[t0] = jk(f[f"meff_jk_t0{t0}"][()])     # (pz:7, w:10, t:20), (pz:7, w:10, t:20)
pdf = PdfPages(os.path.join(save_dir, "plots", f"S4_sgevp_fit_svd{check_svdcut:g}_{run_tag}.pdf"))
for pz in sgevp_window_fixed:
    ipz = pz_list.index(pz)
    first, last = sgevp_window_fixed[pz]
    for t0 in sgevp_t0[pz]:
        d = np.load(os.path.join(save_dir, sgevp_fit_name.format(t0=t0, pz=pz, svdcut=check_svdcut, first=first, last=last)))
        A = d["M_jk"]                               # (jk:1504, w:10)
        B = d["B_jk"]                               # (jk:1504, w:10)
        dE = d["dE_jk"]                             # (jk:1504)
        pull = d["pull"].reshape(A.shape[1], -1)    # (w:10, t:n)
        t_points = np.arange(1, check_t_max + 1)    # (t:14)
        t_fine = np.linspace(first, last, 50)       # (t_fine:50)
        fig, axes = plt.subplots(2, 5, figsize=(16, 6.4), sharex=True)
        for iw in range(A.shape[1]):
            ax = axes.flat[iw]
            m = meff_mean[t0][ipz, iw, t_points]
            e = meff_err[t0][ipz, iw, t_points]
            ok = np.isfinite(m) & (e < check_err_max)
            inside = ok & (t_points >= first) & (t_points <= last)
            outside = ok & ~inside
            ax.errorbar(t_points[inside], m[inside], e[inside], color="#2a78d6", marker="o", ms=3.5, ls="none", elinewidth=0.8, capsize=1.5)
            ax.errorbar(t_points[outside], m[outside], e[outside], color="#2a78d6", marker="o", mfc="none", ms=3.5, ls="none", elinewidth=0.8, capsize=1.5)
            curve = A[:, iw, None] + B[:, iw, None] * t_fine * np.exp(-dE[:, None] * t_fine)    # (jk:1504, t_fine:50)
            c_mean, c_err = jk(curve)                                                           # (t_fine:50), (t_fine:50)
            ax.fill_between(t_fine, c_mean - c_err, c_mean + c_err, color="#2a78d6", alpha=0.25, lw=0)
            A_mean, A_err = jk(A[:, iw])
            ax.axhspan(A_mean - A_err, A_mean + A_err, color="#898781", alpha=0.3, lw=0)
            ax.grid(color="#e1e0d9", lw=0.5)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_title(f"w = {iw}:  M = {gv.gvar(A_mean, A_err)},  mean pull {pull[iw].mean():+.2f}", fontsize=8)
            ax.tick_params(labelsize=7)
        for ax in axes[1]:
            ax.set_xlabel("t", fontsize=8)
        dE_mean, dE_err = jk(dE)
        fig.suptitle(f"{plot_title}: sGEVP fit, pz {pz}, t0 {t0}, window {first}-{last}, svdcut {check_svdcut:g};  chi2/dof {float(d['chi2']) / int(d['dof']):.3f}, "
                     f"dE {gv.gvar(dE_mean, dE_err)};  filled: fitted points", fontsize=10)
        fig.tight_layout()
        save_figure(fig, pdf, "S4_sgevp_fit", f"sgevp_fit_pz{pz}_t0{t0}")
        print(f"pz {pz}, t0 {t0}: mean pull per w " + " ".join(f"{p:+.2f}" for p in pull.mean(axis=1)))
pdf.close()

## S5. Matrix elements of the sGEVP fit: AIC over the windows, average over $t_0$

In [ ]:
%%only_for sgevp
#======== S5. matrix elements of the sGEVP fit:  M(pz, w) and dE, AIC over the windows and average over t0;  one file per variant and svdcut ========
#variants:  "fixed" = the fixed window of each pz at every t0 of the pz;  "B1", "B2" = every window of that grid at every t0 (only if S3 fitted the B1 grid)
#in every jackknife sample:  weight ~ exp[-(chi2_j + 2k + 2 N_cut)/2] over all (t0, window) of the variant (FIT_PLAN.md sections 7, 9, 11);
#  N_cut = 10 w x the t of the union of the variant's windows that this window leaves out;  "fixed": only the t0 differ, so this is the t0 average
#nothing is averaged over svdcuts
for svdcut in sgevp_svdcuts:
    for variant in sgevp_variants:
        M_all = []                      #M_all[pz]: (jk:1504, w:10)
        dE_all = []                     #dE_all[pz]: (jk:1504)
        weights = {}                    #weights[pz]: (jk:1504, choice:n)
        choices = {}                    #choices[pz]: (choice:n, 3) = t0, first t, last t
        for pz in sgevp_window_fixed:
            windows = sgevp_window_list[pz][variant]
            union = set()               #the t of the union of the windows of this variant
            for first, last in windows:
                union = union | set(range(first, last + 1))
            M_choice = []
            dE_choice = []
            chi2_choice = []
            k = []
            n_cut = []
            choices[pz] = []
            for t0 in sgevp_t0[pz]:
                for first, last in windows:
                    file = os.path.join(save_dir, sgevp_fit_name.format(t0=t0, pz=pz, svdcut=svdcut, first=first, last=last))
                    assert os.path.exists(file), f"no fit {file}: run S3"
                    d = np.load(file)
                    assert str(d["settings"]) == sgevp_settings_text, f"{file} was fitted with other settings than S0"
                    M_choice.append(d["M_jk"])              # (jk:1504, w:10)
                    dE_choice.append(d["dE_jk"])            # (jk:1504)
                    chi2_choice.append(d["chi2_jk"])        # (jk:1504)
                    k.append(int(d["k"]))
                    n_cut.append(d["M_jk"].shape[1] * (len(union) - (last - first + 1)))
                    choices[pz].append((t0, first, last))
            M_choice = np.stack(M_choice, axis=1)           # (jk:1504, choice:n, w:10)
            dE_choice = np.stack(dE_choice, axis=1)         # (jk:1504, choice:n)
            chi2_choice = np.stack(chi2_choice, axis=1)     # (jk:1504, choice:n)
            M, dE, weights[pz] = aic_average(M_choice, dE_choice, chi2_choice, np.array(k), np.array(n_cut))
            M_all.append(M)
            dE_all.append(dE)
        M_all = np.stack(M_all, axis=2)                     # (jk:1504, w:10, pz:7)
        dE_all = np.stack(dE_all, axis=1)                   # (jk:1504, pz:7)
        #---- save
        file = os.path.join(save_dir, sgevp_result_name.format(variant=variant, svdcut=svdcut))
        if os.path.exists(file):
            print("replacing", file)
        with h5py.File(file, "w") as f:
            f["M_jk"] = M_all                               # (jk:1504, w:10, pz:7)
            f["dE_jk"] = dE_all                             # (jk:1504, pz:7)
            for pz in sgevp_window_fixed:
                f[f"pz{pz}/choices"] = np.array(choices[pz])        # (choice:n, 3)  t0, first t, last t
                f[f"pz{pz}/weight_jk"] = weights[pz]                # (jk:1504, choice:n)
            f["settings"] = sgevp_settings_text
            f["pz_list"] = list(sgevp_window_fixed)
            with h5py.File(os.path.join(data_save_dir, meff_name), "r") as g:
                f["cfg_list"] = g["cfg_list"][()]           # (cfg:1504)  sample j leaves out cfg j
        #---- print
        M0_mean, M0_err = jk(M_all[:, 0])                   # (pz:7), (pz:7)
        dE_mean, dE_err = jk(dE_all)                        # (pz:7), (pz:7)
        print(f"\nsGEVP, {variant}, svdcut {svdcut:g}:")
        print("   M(w 0) pz 0-6: " + " ".join(str(gv.gvar(m, e)) for m, e in zip(M0_mean, M0_err)))
        print("   dE     pz 0-6: " + " ".join(str(gv.gvar(m, e)) for m, e in zip(dE_mean, dE_err)))
        for pz in sgevp_window_fixed:
            weight_mean = weights[pz].mean(axis=0)          # (choice:n)
            top = np.argsort(-weight_mean)[:3]
            text = []
            for i in top:
                t0, first, last = choices[pz][i]
                text.append(f"t0 {t0} window {first}-{last}: {weight_mean[i]:.2f}")
            print(f"   pz {pz}: largest weights " + ",  ".join(text))
        print("saved", file)

## I1. Double ratio and single ratio of every model

In [ ]:
#======== I1. double ratio and single ratio of every model whose results exist (R5, S5);  one file per model ========
#double ratio(pz, w) = [M(pz, w) / M(pz, 0)] / [M(0, w) / M(0, 0)]  in every jackknife sample;  the reference is pz 0 of the same frame
#single ratio(pz, w) = M(pz, w) / M(0, w) x E0(0) / E0(pz)  in every jackknife sample, E0 from matrix_element_analysis_final_2pt.ipynb (forward frame only)
itd_ratio_models = ["separate", "shared"]       #the ratio results that are used (R5)
itd_ratio_svdcut = 1e-2
itd_sgevp_variant = "fixed"                     #the sGEVP result that is used (S5):  "fixed", "B1" or "B2"
itd_sgevp_svdcut = 1e-2
itd_E0_file = os.path.join(analysis_dir, f"{hadron}_forward_ncfg{ncfg}", "results", f"pt2_aic_E0_gap_forward_q0_0_0_ncfg{ncfg}_{hadron_tag}.h5")

#---- the results files of the models
result_files = {}               #result_files[model name]: the file of R5 or S5
for model in itd_ratio_models:
    result_files[f"ratio_{model}"] = os.path.join(save_dir, ratio_result_name.format(model=model, svdcut=itd_ratio_svdcut))
result_files["sgevp"] = os.path.join(save_dir, sgevp_result_name.format(variant=itd_sgevp_variant, svdcut=itd_sgevp_svdcut))
models = []
for name in result_files:
    if os.path.exists(result_files[name]):
        models.append(name)
    else:
        print(f"{name}: no file {result_files[name]}, left out")
assert len(models) > 0, "no results at all: run R5 or S5 first"

#---- E0 for the single ratio (forward frame)
E0_jk = None
if frame == "forward" and os.path.exists(itd_E0_file):
    with h5py.File(itd_E0_file, "r") as f:
        E0_jk = f["projected/E0_jk"][()]                                    # (jk:1504, pz:7)
    print("E0 from", itd_E0_file)
elif frame == "forward":
    print(f"no E0 file {itd_E0_file}:  the single ratio is left out (run matrix_element_analysis_final_2pt.ipynb for it)")

#---- double ratio and single ratio of every model
double_ratio = {}               #double_ratio[model]: (jk:1504, w:10, pz:7)
single_ratio = {}               #single_ratio[model]: (jk:1504, w:10, pz:7)
for name in models:
    with h5py.File(result_files[name], "r") as f:
        M = f["M_jk"][()]                                                   # (jk:1504, w:10, pz:7)
        pz_list = f["pz_list"][()]                                          # (pz:7)
        cfg_result = f["cfg_list"][()]                                      # (cfg:1504)
    n_w = M.shape[1]
    double_ratio[name] = (M / M[:, :1]) / (M[:, :, :1] / M[:, :1, :1])      # (jk:1504, w:10, pz:7)
    if E0_jk is not None:
        with h5py.File(itd_E0_file, "r") as f:
            if "cfg_list" in f:
                assert np.array_equal(f["cfg_list"][()], cfg_result), f"the E0 file has other cfgs than {result_files[name]}"
        single_ratio[name] = M / M[:, :, :1] * (E0_jk[:, None, :1] / E0_jk[:, None, :])     # (jk:1504, w:10, pz:7)
    nu = 2 * np.pi / Ls * np.outer(np.arange(n_w), pz_list)                 # (w:10, pz:7)
    #---- save
    file = os.path.join(save_dir, itd_name.format(model=name))
    if os.path.exists(file):
        print("replacing", file)
    with h5py.File(file, "w") as f:
        f["double_ratio_jk"] = double_ratio[name]                           # (jk:1504, w:10, pz:7)
        if name in single_ratio:
            f["single_ratio_jk"] = single_ratio[name]                   # (jk:1504, w:10, pz:7)
            f["E0_file"] = itd_E0_file
        f["M_jk"] = M                                                       # (jk:1504, w:10, pz:7)
        f["nu"] = nu                                                        # (w:10, pz:7)
        f["pz_list"] = pz_list
        f["result_file"] = result_files[name]
        f["cfg_list"] = cfg_result
    #---- print
    print(f"\n{name}:  double ratio(pz, w), rows pz 1-{len(pz_list) - 1}, columns w 1-{n_w - 1}")
    mean, err = jk(double_ratio[name])                                      # (w:10, pz:7), (w:10, pz:7)
    for ipz in range(1, len(pz_list)):
        print(f"   pz {pz_list[ipz]}: " + " ".join(f"{str(gv.gvar(mean[w, ipz], err[w, ipz])):>11s}" for w in range(1, n_w)))
    if name in single_ratio:
        print(f"{name}:  single ratio(pz, w), rows pz 1-{len(pz_list) - 1}, columns w 0-{n_w - 1}")
        mean, err = jk(single_ratio[name])                                  # (w:10, pz:7), (w:10, pz:7)
        for ipz in range(1, len(pz_list)):
            print(f"   pz {pz_list[ipz]}: " + " ".join(f"{str(gv.gvar(mean[w, ipz], err[w, ipz])):>11s}" for w in range(n_w)))
    print("saved", file)

#---- the largest difference of the double ratio between two models, in units of the larger of their two errors
if len(models) > 1:
    print(f"\nlargest |double ratio(model 1) - double ratio(model 2)| / larger error, over every pair of {models}:  rows pz 1-{len(pz_list) - 1}, columns w 1-{n_w - 1}")
    for ipz in range(1, len(pz_list)):
        line = []
        for w in range(1, n_w):
            largest = 0
            for i1 in range(len(models)):
                for i2 in range(i1 + 1, len(models)):
                    m1, e1 = jk(double_ratio[models[i1]][:, w, ipz])
                    m2, e2 = jk(double_ratio[models[i2]][:, w, ipz])
                    largest = max(largest, abs(m1 - m2) / max(e1, e2))
            line.append(f"{largest:5.2f}")
        print(f"   pz {pz_list[ipz]}: " + " ".join(line))

## I2. Plots of the double ratio and the single ratio: ratio fits, sGEVP, or both

In [ ]:
#======== I2. plots of the double ratio and the single ratio against nu = 2 pi pz w / Ls:  the ratio fits, the sGEVP, or both ========
itd_plot = "both"                               #"ratio": the ratio fits only;  "sgevp": the sGEVP only;  "both": all of them together
itd_plot_quantities = ["double_ratio", "single_ratio"]     #the single ratio is left out where it was not made (sym frame, no E0 file)
itd_plot_layout = ["per_pz", "all_pz"]          #"per_pz": one panel per pz 1-6, the models side by side;  "all_pz": one panel per model, all pz together
itd_plot_err_max = 0.5                          #points with a larger error are left out (and listed on the plot)
itd_ylim = {"double_ratio": (-0.3, 1.4), "single_ratio": (-0.1, 1.4)}
model_label = {"ratio_separate": "ratio fit, separate fraction", "ratio_shared": "ratio fit, shared fraction", "sgevp": "sGEVP M_eff fit"}
model_color = {"ratio_separate": "#2a78d6", "ratio_shared": "#eb6834", "sgevp": "#1baf7a"}
model_marker = {"ratio_separate": "o", "ratio_shared": "s", "sgevp": "^"}
pz_colors = ["#52514e", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
pz_markers = ["o", "s", "^", "D", "v", "P", "X"]

#---- the models to plot, read from the files of I1
assert itd_plot in ["ratio", "sgevp", "both"], f"itd_plot = {itd_plot}: use ratio, sgevp or both"
plot_models = []
for name in ["ratio_separate", "ratio_shared", "sgevp"]:
    wanted = itd_plot == "both" or name.startswith(itd_plot)
    if wanted and os.path.exists(os.path.join(save_dir, itd_name.format(model=name))):
        plot_models.append(name)
assert len(plot_models) > 0, f"nothing to plot for itd_plot = {itd_plot}: run I1 (and R5 or S5) first"
values = {}                     #values[(quantity, model)]: jackknife mean and error, each (w:10, pz:7)
for name in plot_models:
    with h5py.File(os.path.join(save_dir, itd_name.format(model=name)), "r") as f:
        nu = f["nu"][()]                                                    # (w:10, pz:7)
        pz_list = f["pz_list"][()]                                          # (pz:7)
        for quantity in itd_plot_quantities:
            if f"{quantity}_jk" in f:
                values[(quantity, name)] = jk(f[f"{quantity}_jk"][()])
n_w = nu.shape[0]

pdf = PdfPages(os.path.join(save_dir, "plots", f"I2_itd_{itd_plot}_{run_tag}.pdf"))
for quantity in itd_plot_quantities:
    names = []                  #the models that have this quantity
    for name in plot_models:
        if (quantity, name) in values:
            names.append(name)
    if len(names) == 0:
        print(f"{quantity.replace('_', ' ')}: not made for these models, left out")
        continue
    #---- one panel per pz, the models side by side
    if "per_pz" in itd_plot_layout:
        fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharey=True)
        for ipz in range(1, len(pz_list)):
            ax = axes.flat[ipz - 1]
            step = 2 * np.pi / Ls * pz_list[ipz]                            #the nu step between two w
            left_out = []
            for i_model, name in enumerate(names):
                mean, err = values[(quantity, name)]
                ok = err[:, ipz] < itd_plot_err_max
                shift = (i_model - (len(names) - 1) / 2) * 0.18 * step      #a small shift in nu, so that the models do not sit on top of each other
                ax.errorbar(nu[ok, ipz] + shift, mean[ok, ipz], err[ok, ipz], color=model_color[name], marker=model_marker[name], ms=4,
                            ls="none", elinewidth=0.9, capsize=1.5, label=model_label[name])
                if (~ok).any():
                    left_out.append(f"{name}: w {' '.join(str(w) for w in np.where(~ok)[0])}")
            if left_out:
                ax.text(0.98, 0.97, "left out:\n" + "\n".join(left_out), transform=ax.transAxes, ha="right", va="top", fontsize=6.5)
            ax.axhline(1, color="#898781", lw=0.6)
            ax.axhline(0, color="#c3c2b7", lw=0.5)
            ax.set_ylim(*itd_ylim[quantity])
            ax.set_title(f"pz = {pz_list[ipz]}", fontsize=10)
            ax.grid(color="#e1e0d9", lw=0.5)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=8)
            if ipz >= 4:
                ax.set_xlabel(r"$\nu = 2\pi p_z w / L_s$", fontsize=9)
            if ipz in (1, 4):
                ax.set_ylabel(quantity.replace("_", " "), fontsize=9)
        axes.flat[0].legend(fontsize=8, frameon=False, loc="lower left")
        fig.suptitle(f"{plot_title}: {quantity.replace('_', ' ')};  points with error > {itd_plot_err_max} left out", fontsize=10)
        fig.tight_layout()
        save_figure(fig, pdf, "I2_itd", f"{quantity}_per_pz_{itd_plot}")
    #---- one panel per model, all pz together
    if "all_pz" in itd_plot_layout:
        fig, axes = plt.subplots(1, len(names), figsize=(4.6 * len(names), 4.2), sharey=True, squeeze=False)
        for i_model, name in enumerate(names):
            ax = axes[0, i_model]
            mean, err = values[(quantity, name)]
            for ipz in range(1, len(pz_list)):
                ok = err[:, ipz] < itd_plot_err_max
                ax.errorbar(nu[ok, ipz], mean[ok, ipz], err[ok, ipz], color=pz_colors[ipz], marker=pz_markers[ipz], ms=3.5,
                            ls="none", elinewidth=0.8, capsize=1.5, label=f"pz = {pz_list[ipz]}")
            ax.axhline(1, color="#898781", lw=0.6)
            ax.axhline(0, color="#c3c2b7", lw=0.5)
            ax.set_ylim(*itd_ylim[quantity])
            ax.set_title(model_label[name], fontsize=10)
            ax.set_xlabel(r"$\nu = 2\pi p_z w / L_s$", fontsize=9)
            ax.grid(color="#e1e0d9", lw=0.5)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=8)
        axes[0, 0].set_ylabel(quantity.replace("_", " "), fontsize=9)
        axes[0, 0].legend(fontsize=7, frameon=False, ncol=2)
        fig.suptitle(f"{plot_title}: {quantity.replace('_', ' ')};  points with error > {itd_plot_err_max} left out", fontsize=10)
        fig.tight_layout()
        save_figure(fig, pdf, "I2_itd", f"{quantity}_all_pz_{itd_plot}")
pdf.close()